Load submission summary clinvar table (detailed)

In [4]:
import pandas as pd



In [5]:
# --- Load submission summary ---
submission_summary_path = 'submission_summary.txt.gz'

# Load starting from the actual data row
# skiprows=18 skips the first 18 junk lines
# header=0 tells it the next line (line 19) is the header
df = pd.read_csv(submission_summary_path, 
                 sep='\t', 
                 compression='gzip', 
                 skiprows=18, 
                 low_memory=False)

# Check the columns to ensure they are what you expect
print(df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'submission_summary.txt.gz'

Load deltas of BED columns from parquet, use variant summary from clinvar to get #VariationID

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

# --- 1. Identify the columns to load ---
parquet_file = pq.ParquetFile("clinvar_full_deltas.parquet")

# Get all column names from the schema
all_columns = parquet_file.schema.names

# Filter: We want the index columns + any column starting with "D_BED_"
index_cols = ['chrom', 'pos', 'ref', 'alt', 'label']
delta_cols = [c for c in all_columns if c.startswith('D_BED_')]
cols_to_load = index_cols + delta_cols

print(f"Loading {len(cols_to_load)} columns (Index + {len(delta_cols)} BED features)...")

# --- 2. Load ONLY those columns from the full file ---
# Note: This reads the whole file. If it's too big for RAM, use iter_batches()
sample_df = parquet_file.read(columns=cols_to_load).to_pandas()

# --- 3. Load ClinVar variant_summary (Optimized) ---
variant_summary_path = "variant_summary.txt.gz"

# Only load columns needed for the mapping to save memory
clinvar_cols = ["Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF", "VariationID", "Type", "GeneSymbol"]

var_df_summary = pd.read_csv(
    variant_summary_path,
    sep="\t",
    compression="gzip",
    usecols=clinvar_cols, 
    low_memory=False
)

# --- 4. Prepare ClinVar Data ---
# Rename to match your parquet schema for easy merging
var_df_summary.rename(columns={
    "Chromosome": "chrom", 
    "PositionVCF": "pos", 
    "ReferenceAlleleVCF": "ref", 
    "AlternateAlleleVCF": "alt",
    "VariationID": "#VariationID"
}, inplace=True)

# Normalize Chromosome column (Ensure "chr" prefix matches your Parquet data)
# Check if your Parquet uses "chr1" or just "1". Adjust accordingly.
# Assuming Parquet uses "chr1" and ClinVar uses "1":
var_df_summary["chrom"] = var_df_summary["chrom"].astype(str)
if not var_df_summary["chrom"].iloc[0].startswith("chr"):
    var_df_summary["chrom"] = "chr" + var_df_summary["chrom"]

# Filter for SNVs only
snp_summary = var_df_summary[var_df_summary["Type"] == "single nucleotide variant"].copy()

# Ensure integer types for position
sample_df["pos"] = pd.to_numeric(sample_df["pos"], errors="coerce").astype("Int64")
snp_summary["pos"] = pd.to_numeric(snp_summary["pos"], errors="coerce").astype("Int64")

# --- 5. Merge and Save ---
print("Merging data...")
merged_df = sample_df.merge(
    snp_summary[["chrom", "pos", "ref", "alt", "#VariationID","GeneSymbol" ]],
    on=["chrom", "pos", "ref", "alt"],
    how="left"
)

# Check how many IDs were found
print(f"Matched {merged_df['#VariationID'].notna().sum():,} variants to ClinVar IDs.")

merged_df.to_parquet("clinvar_full_deltas_with_snp_ids.parquet", index=False)
print("Done.")


Loading 26 columns (Index + 21 BED features)...
Merging data...
Matched 28,096 variants to ClinVar IDs.
Done.


In [189]:
merged_df = pd.read_parquet("clinvar_full_deltas_with_snp_ids.parquet")

In [190]:
merged_df

,chrom,pos,ref,alt,label,D_BED_protein_coding_gene,D_BED_lncRNA,D_BED_exon,D_BED_intron,D_BED_splice_donor,...,D_BED_5UTR-,D_BED_3UTR+,D_BED_3UTR-,D_BED_skipped_exon,D_BED_always_on_exon,D_BED_start_codon,D_BED_stop_codon,D_BED_ORF,#VariationID,GeneSymbol
0,chr1,69511,A,G,Benign,0.000813,0.001542,0.007079,-0.000323,0.001979,...,0.000112,-0.001051,0.002359,0.002956,0.007551,0.002692,-0.010524,0.002611,NaN,None
1,chr1,953279,T,C,Benign,0.002615,0.003673,-0.005527,0.006544,0.001130,...,0.001267,0.000200,0.001674,0.005859,-0.006019,-0.004770,-0.013489,-0.005872,NaN,None
2,chr1,973858,G,C,Benign,-0.004761,-0.017012,-0.000898,-0.001429,0.056321,...,-0.000327,-0.001616,-0.003181,0.012929,-0.006858,-0.014206,0.002861,0.010388,NaN,None
3,chr1,973929,T,C,Benign,-0.002591,-0.011274,-0.018084,0.026183,0.022590,...,-0.000403,-0.000111,-0.002868,-0.005005,-0.020452,-0.000382,-0.010587,-0.007649,NaN,None
4,chr1,978953,C,G,Benign,0.000017,0.001085,-0.001471,0.004100,0.001751,...,-0.004799,-0.002355,0.002307,-0.003560,-0.005361,-0.003455,-0.005807,-0.003341,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41210,chrY,2787426,C,G,Pathogenic,0.021540,0.005181,0.028580,0.001996,0.001023,...,0.011704,-0.000144,-0.005848,0.003274,0.024705,0.006304,0.001164,0.063287,9739.0,SRY
41211,chrY,2787515,C,A,Pathogenic,0.004168,-0.021232,-0.140918,0.010377,-0.006219,...,0.014050,0.001785,0.001331,-0.004686,-0.129609,0.004140,0.020241,-0.111509,492908.0,SRY
41212,chrY,2787551,C,T,Pathogenic,-0.023064,0.029417,0.004175,-0.007124,-0.002365,...,-0.010652,0.000325,-0.000694,0.012869,0.005396,0.031488,0.006155,0.026616,9754.0,SRY
41213,chrY,7063898,A,T,Pathogenic,0.022077,-0.148108,-0.012861,-0.025645,-0.001313,...,0.000405,0.003400,-0.002748,-0.042884,-0.010173,-0.136555,0.001420,-0.009295,625467.0,TBL1Y


Filter the variants to only those with valid #VariationID

In [45]:
merged_df["#VariationID"] = (
    merged_df["#VariationID"]
    .astype("Int64")   # capital I
)

# only mapped variants
filtered_df = merged_df[merged_df["#VariationID"].notna()]

Fetch rationales from the submission clinvar table

In [ ]:

import pyarrow.parquet as pq

import pandas as pd

def fetch_rationales_combined(mapped_df, submission_summary_path, sub_df=None):
    """
    Fetches rationales, prioritizing 'Description' for coverage
    but adding 'ExplanationOfInterpretation' for depth.

    Skips any rows in mapped_df where #VariationID is NaN (won't attempt grouping/merge for them).
    """
    cols_to_use = ['#VariationID', 'Description', 'ExplanationOfInterpretation']

    print("Loading submission summary...")
    if sub_df is None:
        sub_df = pd.read_csv(
            submission_summary_path,
            sep='\t',
            compression='gzip',
            usecols=cols_to_use,
            skiprows=18,
            low_memory=False
        )

    # --- helper ---
    def clean_text(text_series):
        valid = [str(t) for t in text_series.dropna() if str(t).strip() not in ['-', '']]
        # stable unique order (instead of set(), which can scramble)
        seen = set()
        uniq = []
        for v in valid:
            if v not in seen:
                uniq.append(v)
                seen.add(v)
        return ' | '.join(uniq)

    # Drop NaN IDs from submission summary before groupby (safety)
    sub_df = sub_df.dropna(subset=['#VariationID']).copy()

    print("Aggregating rationales...")
    grouped = (
        sub_df.groupby('#VariationID', as_index=False)
              .agg({
                  'Description': clean_text,
                  'ExplanationOfInterpretation': clean_text
              })
    )

    def combine_columns(row):
        parts = []
        if row.get('Description'):
            parts.append(f"DESC: {row['Description']}")
        if row.get('ExplanationOfInterpretation'):
            parts.append(f"EXPL: {row['ExplanationOfInterpretation']}")
        return "\n".join(parts) if parts else "No detailed rationale provided."

    grouped['FullRationale'] = grouped.apply(combine_columns, axis=1)
    grouped = grouped[['#VariationID', 'FullRationale']]

    # --- IMPORTANT PART: only merge for rows that actually have an ID ---
    print("Merging with variant data (skipping NaN #VariationID rows)...")

    final_df = mapped_df.copy()
    final_df['FullRationale'] = "No rationale provided."  # default for everyone

    mask_has_id = final_df['#VariationID'].notna()
    final_df.loc[mask_has_id, 'FullRationale'] = (
        final_df.loc[mask_has_id]
                .merge(grouped, on='#VariationID', how='left')['FullRationale_y']
                .fillna("No rationale provided.")
                .to_numpy()
    )

    return final_df


# Usage
final_df = fetch_rationales_combined(merged_df, 'submission_summary.txt.gz')

new_cols = final_df.columns[0:5].tolist() + ["GeneSymbol"] + ["FullRationale"] + ["#VariationID"] + [c for c in final_df.columns[5:] if c not in ["GeneSymbol", "FullRationale", "#VariationID"]]
final_df = final_df[new_cols]

# SongLab Consequence
print("Merging SongLab annotations...")
try:
    songlab = pd.read_parquet("hf://datasets/songlab/clinvar_vs_benign/test.parquet")
    # Ensure chromosome format matches (e.g., "chr1")
    songlab["chrom"] = "chr" + songlab["chrom"].astype(str).str.replace("chr", "") 
    
    # Merge
    merged = pd.merge(final_df, songlab, on=["chrom", "pos", "ref", "alt"], how="left")
    final_df["sognlab_consequence"] = merged["consequence"]

    ordered_cols = final_df.columns[:8].tolist() + ["sognlab_consequence"] + [c for c in final_df.columns[8:] if c not in ["sognlab_consequence"]]
    final_df = final_df[ordered_cols]
except Exception as e:
    print(f"Warning: Could not load SongLab dataset ({e}). Skipping that column.")
    final_df["sognlab_consequence"] = "Not Available"

# region obtaining
parquet_file = pq.ParquetFile("clinvar_enriched.parquet")

sample_df = parquet_file.read_row_group(0).to_pandas()
final_df = pd.merge(final_df, sample_df[["region","chrom","pos","ref","alt"]], on=["chrom","pos","ref","alt"], how="left")



final_df.to_csv("all_variants_with_rationales.csv")

Loading submission summary...
Aggregating rationales...
Merging with variant data (skipping NaN #VariationID rows)...
Merging SongLab annotations...


In [197]:
final_df.to_csv("all_variants_with_rationales.csv")

claude-based mechanism assignments + llm prompts sampling

In [231]:
import pandas as pd
import numpy as np

# =============================================================================
# 1. HYPOTHESIS LOGIC
# =============================================================================

NT_HYPOTHESIS_LOGIC = {
    # ==========================================================================
    # TIER 0: DEFINITIVE LOSS-OF-FUNCTION (Priority 0)
    # ==========================================================================
    
    "SPLICE_SITE_DESTROYED": {
        "primary": [
            {"feature": "D_BED_splice_donor", "threshold": -0.5, "direction": "negative"},
            {"feature": "D_BED_splice_acceptor", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "HIGH",
        "description": "Loss of canonical splice donor or acceptor site",
        "priority": 0
    },
    
    "START_LOSS": {
        "primary": [
            {"feature": "D_BED_start_codon", "threshold": -0.4, "direction": "negative"}  # Lowered from -0.5
        ],
        "secondary": [],
        "confidence": "HIGH",
        "description": "Loss of translation initiation codon",
        "priority": 0
    },
    
    "STOP_CODON_DISRUPTION": {
        "primary": [
            {"feature": "D_BED_stop_codon", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "HIGH",
        "description": "Disruption of stop codon causing read-through",
        "priority": 0
    },

    # ==========================================================================
    # TIER 1: LIKELY FUNCTIONAL IMPACT (Priority 1)
    # ==========================================================================
    
    # --- Splice-related mechanisms ---
    
    "CRYPTIC_SPLICE_GAIN": {
        "primary": [
            {"feature": "D_BED_splice_donor", "threshold": 0.5, "direction": "positive"},
            {"feature": "D_BED_splice_acceptor", "threshold": 0.5, "direction": "positive"}
        ],
        "secondary": [],
        "confidence": "MEDIUM_HIGH",
        "description": "Creation of new cryptic splice site",
        "priority": 1
    },
    
    "SPLICE_INDUCED_EXONIZATION": {
        "primary": [
            {"feature": "D_BED_exon", "threshold": 0.4, "direction": "positive"}
        ],
        "secondary": [
            {"feature": "D_BED_intron", "threshold": -0.3, "direction": "negative"},
            {"feature": "D_BED_ORF", "threshold": 0.3, "direction": "positive"}
        ],
        "confidence": "MEDIUM_HIGH",
        "description": "Intronic sequence included as coding exon, typically from splice site disruption",
        "priority": 1
    },
    
    "SPLICE_INDUCED_INTRON_RETENTION": {
        "primary": [
            {"feature": "D_BED_intron", "threshold": 0.5, "direction": "positive"}
        ],
        "secondary": [
            {"feature": "D_BED_splice_donor", "threshold": -0.3, "direction": "negative"}
        ],
        "confidence": "MEDIUM_HIGH",
        "description": "Intron retention due to splice site weakening",
        "priority": 1
    },
    
    "EXON_SKIPPING_INDUCED": {
        "primary": [
            {"feature": "D_BED_skipped_exon", "threshold": 0.4, "direction": "positive"}
        ],
        "secondary": [
            {"feature": "D_BED_always_on_exon", "threshold": -0.3, "direction": "negative"}
        ],
        "confidence": "MEDIUM_HIGH",
        "description": "Induction of exon skipping",
        "priority": 1
    },
    
    # --- Exon/coding loss mechanisms ---
    
    "CONSTITUTIVE_EXON_LOSS": {
        "primary": [
            {"feature": "D_BED_always_on_exon", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "MEDIUM_HIGH",
        "description": "Loss of constitutively included exon",
        "priority": 1
    },
    
    "STRONG_CODING_LOSS": {
        "primary": [
            {"feature": "D_BED_exon", "threshold": -0.45, "direction": "negative"}
        ],
        "secondary": [
            {"feature": "D_BED_ORF", "threshold": -0.3, "direction": "negative"}
        ],
        "confidence": "MEDIUM_HIGH",
        "description": "Loss of coding exon identity with ORF disruption",
        "priority": 1
    },
    
    "STRONG_EXON_LOSS": {
        "primary": [
            {"feature": "D_BED_exon", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Strong loss of exon identity",
        "priority": 1
    },
    
    # --- ORF mechanisms ---
    
    "ORF_DISRUPTION": {
        "primary": [
            {"feature": "D_BED_ORF", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Disruption of open reading frame",
        "priority": 1
    },
    
    "CRYPTIC_EXON_INCLUSION": {
        "primary": [
            {"feature": "D_BED_skipped_exon", "threshold": 0.5, "direction": "positive"}
        ],
        "secondary": [
            {"feature": "D_BED_exon", "threshold": 0.4, "direction": "positive"}
        ],
        "confidence": "MEDIUM_HIGH",
        "description": "Inclusion of cryptic/normally-skipped exon into transcript",
        "priority": 1
    },

    # ==========================================================================
    # TIER 2: REGULATORY & UTR IMPACT (Priority 2)
    # ==========================================================================
    
    # --- Start codon compound mechanisms ---
    
    "START_LOSS_WITH_UTR_EXTENSION": {
        "primary": [
            {"feature": "D_BED_start_codon", "threshold": -0.35, "direction": "negative"}
        ],
        "secondary": [
            {"feature": "D_BED_5UTR+", "threshold": 0.3, "direction": "positive"}
        ],
        "confidence": "MEDIUM_HIGH",
        "description": "Start codon loss with consequent 5'UTR extension",
        "priority": 2
    },
    
    "START_LOSS_WITH_ORF_DISRUPTION": {
        "primary": [
            {"feature": "D_BED_start_codon", "threshold": -0.35, "direction": "negative"}
        ],
        "secondary": [
            {"feature": "D_BED_ORF", "threshold": -0.3, "direction": "negative"}
        ],
        "confidence": "MEDIUM_HIGH",
        "description": "Start codon loss with downstream ORF disruption",
        "priority": 2
    },
    
    # --- Regulatory element destruction ---
    
    "PROMOTER_DESTRUCTION": {
        "primary": [
            {"feature": "D_BED_promoter_Tissue_specific", "threshold": -0.5, "direction": "negative"},
            {"feature": "D_BED_promoter_Tissue_invariant", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Destruction of promoter element",
        "priority": 2
    },
    
    "ENHANCER_DESTRUCTION": {
        "primary": [
            {"feature": "D_BED_enhancer_Tissue_specific", "threshold": -0.5, "direction": "negative"},
            {"feature": "D_BED_enhancer_Tissue_invariant", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Destruction of enhancer element",
        "priority": 2
    },
    
    # --- Regulatory element gain ---
    
    "STRONG_REGULATORY_GAIN": {
        "primary": [
            {"feature": "D_BED_enhancer_Tissue_specific", "threshold": 0.6, "direction": "positive"},
            {"feature": "D_BED_promoter_Tissue_specific", "threshold": 0.6, "direction": "positive"}
        ],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Creation of strong regulatory element (potential ectopic expression)",
        "priority": 2
    },
    
    # --- UTR mechanisms ---
    
    "UTR5_DISRUPTION": {
        "primary": [
            {"feature": "D_BED_5UTR+", "threshold": -0.4, "direction": "negative"},
            {"feature": "D_BED_5UTR-", "threshold": 0.4, "direction": "positive"}
        ],
        "secondary": [],
        "confidence": "LOW_MEDIUM",
        "description": "Disruption of 5' UTR structure or regulation",
        "priority": 2
    },
    
    "UTR3_DISRUPTION": {
        "primary": [
            {"feature": "D_BED_3UTR+", "threshold": -0.4, "direction": "negative"},
            {"feature": "D_BED_3UTR-", "threshold": 0.4, "direction": "positive"}
        ],
        "secondary": [],
        "confidence": "LOW_MEDIUM",
        "description": "Disruption of 3' UTR structure or regulation",
        "priority": 2
    },
    
    # --- Other regulatory ---
    
    "POLYA_SIGNAL_LOSS": {
        "primary": [
            {"feature": "D_BED_polyA_signal", "threshold": -0.4, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "LOW_MEDIUM",
        "description": "Loss of polyadenylation signal",
        "priority": 2
    },
    
    "CTCF_BOUNDARY_DISRUPTION": {
        "primary": [
            {"feature": "D_BED_CTCF-bound", "threshold": -0.4, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "LOW_MEDIUM",
        "description": "Disruption of CTCF binding / chromatin boundary element",
        "priority": 2
    },

    # ==========================================================================
    # TIER 3: COMPLEX / RECIPROCAL CHANGES (Priority 3)
    # ==========================================================================
    
    # --- Splice-consequence mechanisms (downstream effects) ---
    
    "CRYPTIC_ORF_GAIN": {
        "primary": [
            {"feature": "D_BED_ORF", "threshold": 0.5, "direction": "positive"}
        ],
        "secondary": [],
        "confidence": "LOW_MEDIUM",
        "description": "Creation of cryptic open reading frame (possible exonization consequence)",
        "priority": 3
    },
    
    "CRYPTIC_ORF_GAIN_WITH_INTRON_LOSS": {
        "primary": [
            {"feature": "D_BED_ORF", "threshold": 0.5, "direction": "positive"}
        ],
        "secondary": [
            {"feature": "D_BED_intron", "threshold": -0.3, "direction": "negative"}
        ],
        "confidence": "MEDIUM",
        "description": "ORF gain with intron loss - likely splice-induced exonization",
        "priority": 3
    },
    
    # --- Exon/intron reciprocal mechanisms ---
    
    "ABERRANT_EXON_INCLUSION": {
        "primary": [
            {"feature": "D_BED_always_on_exon", "threshold": 0.5, "direction": "positive"}
        ],
        "secondary": [
            {"feature": "D_BED_intron", "threshold": -0.3, "direction": "negative"}
        ],
        "confidence": "MEDIUM",
        "description": "Aberrant inclusion of intronic sequence as constitutive exon",
        "priority": 3
    },
    
    "INTRON_RETENTION": {
        "primary": [
            {"feature": "D_BED_intron", "threshold": 0.5, "direction": "positive"}
        ],
        "secondary": [
            {"feature": "D_BED_exon", "threshold": -0.3, "direction": "negative"}
        ],
        "confidence": "MEDIUM",
        "description": "Retention of intron in mature transcript",
        "priority": 3
    },
    
    "RECIPROCAL_EXON_GAIN_INTRON_LOSS": {
        "primary": [
            {"feature": "D_BED_exon", "threshold": 0.4, "direction": "positive"}
        ],
        "secondary": [
            {"feature": "D_BED_intron", "threshold": -0.4, "direction": "negative"}
        ],
        "confidence": "MEDIUM",
        "description": "Exon gain with reciprocal intron loss (exonization)",
        "priority": 3
    },
    
    "RECIPROCAL_EXON_LOSS_INTRON_GAIN": {
        "primary": [
            {"feature": "D_BED_exon", "threshold": -0.4, "direction": "negative"}
        ],
        "secondary": [
            {"feature": "D_BED_intron", "threshold": 0.4, "direction": "positive"}
        ],
        "confidence": "MEDIUM",
        "description": "Exon loss with reciprocal intron gain (intronization)",
        "priority": 3
    },
    
    # --- Gene architecture ---
    
    "GENE_ARCHITECTURE_DISRUPTION": {
        "primary": [
            {"feature": "D_BED_protein_coding_gene", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Disruption of overall gene architecture",
        "priority": 3
    },
    
    "LNCRNA_DISRUPTION": {
        "primary": [
            {"feature": "D_BED_lncRNA", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "LOW",
        "description": "Disruption of lncRNA",
        "priority": 3
    },

    # ==========================================================================
    # TIER 4: WEAK / SUGGESTIVE SIGNALS (Priority 4)
    # ==========================================================================
    
    "WEAK_SPLICE_CHANGE": {
        "primary": [
            {"feature": "D_BED_splice_donor", "threshold": 0.25, "direction": "any"},
            {"feature": "D_BED_splice_acceptor", "threshold": 0.25, "direction": "any"}
        ],
        "secondary": [],
        "confidence": "LOW",
        "description": "Weak change in splice site score (requires further investigation)",
        "priority": 4
    },
    
    "WEAK_REGULATORY_CHANGE": {
        "primary": [
            {"feature": "D_BED_enhancer_Tissue_specific", "threshold": 0.25, "direction": "any"},
            {"feature": "D_BED_promoter_Tissue_specific", "threshold": 0.25, "direction": "any"}
        ],
        "secondary": [],
        "confidence": "LOW",
        "description": "Weak change in regulatory element (requires further investigation)",
        "priority": 4
    },
    
    "WEAK_EXON_CHANGE": {
        "primary": [
            {"feature": "D_BED_exon", "threshold": 0.25, "direction": "any"},
            {"feature": "D_BED_always_on_exon", "threshold": 0.25, "direction": "any"}
        ],
        "secondary": [],
        "confidence": "LOW",
        "description": "Weak change in exon identity (requires further investigation)",
        "priority": 4
    },
    
    "WEAK_ORF_CHANGE": {
        "primary": [
            {"feature": "D_BED_ORF", "threshold": 0.25, "direction": "any"}
        ],
        "secondary": [],
        "confidence": "LOW",
        "description": "Weak change in ORF (requires further investigation)",
        "priority": 4
    },
}

CONFIDENCE_SCORES = {
    "HIGH": 1.0,
    "MEDIUM_HIGH": 0.8,
    "MEDIUM": 0.6,
    "LOW_MEDIUM": 0.4,
    "LOW": 0.2,
    "UNKNOWN": 0.1
}

# =============================================================================
# 2. CORE FUNCTIONS
# =============================================================================

def check_condition(value, threshold, direction):
    """Evaluate a single condition."""
    if pd.isna(value):
        value = 0
    if direction == "negative":
        return value <= threshold
    elif direction == "positive":
        return value >= threshold
    elif direction == "any":
        return abs(value) >= abs(threshold)
    return False


def assign_mechanisms(row, hypothesis_logic):
    """
    Assign ALL matching mechanisms to a variant.
    
    Returns dict with:
        - NT_Mechanism: Primary (highest priority) mechanism
        - NT_All_Mechanisms: All matching mechanisms
        - NT_Mechanism_Count: Number of mechanisms matched
        - Details for each match
    """
    sorted_hypotheses = sorted(hypothesis_logic.items(), key=lambda x: x[1]['priority'])
    
    all_matches = []
    
    for mech_name, rules in sorted_hypotheses:
        # PRIMARY CHECK (OR Logic) - any one must pass
        primary_match = False
        trigger_feature = None
        trigger_value = None
        
        for condition in rules['primary']:
            feature = condition['feature']
            val = row.get(feature, 0)
            if pd.isna(val):
                val = 0
            
            if check_condition(val, condition['threshold'], condition['direction']):
                primary_match = True
                trigger_feature = feature
                trigger_value = val
                break
        
        if not primary_match:
            continue

        # SECONDARY CHECK (AND Logic) - all must pass
        secondary_match = True
        secondary_triggers = []
        
        for condition in rules.get('secondary', []):
            feature = condition['feature']
            val = row.get(feature, 0)
            if pd.isna(val):
                val = 0
            
            if check_condition(val, condition['threshold'], condition['direction']):
                secondary_triggers.append(f"{feature.replace('D_BED_', '')}={val:.3f}")
            else:
                secondary_match = False
                break
        
        if secondary_match:
            all_matches.append({
                'mechanism': mech_name,
                'confidence': rules.get('confidence', 'UNKNOWN'),
                'confidence_score': CONFIDENCE_SCORES.get(rules.get('confidence', 'UNKNOWN'), 0.1),
                'description': rules.get('description', ''),
                'priority': rules['priority'],
                'trigger': f"{trigger_feature.replace('D_BED_', '')}={trigger_value:.3f}",
                'secondary': "; ".join(secondary_triggers) if secondary_triggers else ""
            })
    
    # No matches
    if not all_matches:
        return {
            'NT_Mechanism': 'UNCERTAIN_SIGNIFICANCE',
            'NT_Confidence': 'LOW',
            'NT_Description': 'No strong NT signal detected',
            'NT_Trigger': '',
            'NT_All_Mechanisms': '',
            'NT_Mechanism_Count': 0
        }
    
    # Sort by priority then confidence
    all_matches.sort(key=lambda x: (x['priority'], -x['confidence_score']))
    
    primary = all_matches[0]
    
    # Format trigger with secondary if present
    trigger_str = primary['trigger']
    if primary['secondary']:
        trigger_str += f" [+ {primary['secondary']}]"
    
    # All mechanisms string
    all_mechs_str = "; ".join([
        f"{m['mechanism']}(P{m['priority']},{m['confidence']})" 
        for m in all_matches
    ])
    
    return {
        'NT_Mechanism': primary['mechanism'],
        'NT_Confidence': primary['confidence'],
        'NT_Description': primary['description'],
        'NT_Trigger': trigger_str,
        'NT_All_Mechanisms': all_mechs_str,
        'NT_Mechanism_Count': len(all_matches)
    }


def extract_top_signals(row, delta_cols, k=3):
    """Extract top K signals by magnitude, gain, and loss."""
    data = {}
    for c in delta_cols:
        val = row.get(c, 0)
        data[c] = val if not pd.isna(val) else 0
    
    # Top by absolute magnitude
    sorted_abs = sorted(data.items(), key=lambda x: abs(x[1]), reverse=True)[:k]
    abs_str = "; ".join([f"{k.replace('D_BED_', '')}={v:.3f}" for k, v in sorted_abs])
    
    # Top gains
    gains = [(k, v) for k, v in data.items() if v > 0.01]
    sorted_gains = sorted(gains, key=lambda x: x[1], reverse=True)[:k]
    gain_str = "; ".join([f"{k.replace('D_BED_', '')}={v:.3f}" for k, v in sorted_gains]) if sorted_gains else "None"
    
    # Top losses
    losses = [(k, v) for k, v in data.items() if v < -0.01]
    sorted_losses = sorted(losses, key=lambda x: x[1])[:k]
    loss_str = "; ".join([f"{k.replace('D_BED_', '')}={v:.3f}" for k, v in sorted_losses]) if sorted_losses else "None"
    
    return {
        'Top_Abs_Signals': abs_str,
        'Top_Gain_Signals': gain_str,
        'Top_Loss_Signals': loss_str
    }


# =============================================================================
# 3. SAMPLING FUNCTION FOR LLM EVALUATION
# =============================================================================

def sample_for_llm_evaluation(df, n_per_category=3, random_state=42):
    """
    Sample variants for LLM evaluation, stratified by mechanism and label.
    
    Sampling strategy:
        1. Sample from each NT_Mechanism category
        2. Balance pathogenic vs benign where possible
        3. Prioritize variants WITH ClinVar rationales
    """
    np.random.seed(random_state)
    
    # Filter to variants with rationales
    has_rationale = df['FullRationale'].notna() & (df['FullRationale'] != '') & (df['FullRationale'].str.len() > 10)
    df_with_rationale = df[has_rationale].copy()
    
    print(f"Variants with ClinVar rationales: {len(df_with_rationale)}")
    
    samples = []
    
    # Get unique mechanisms (excluding UNCERTAIN for focused sampling)
    mechanisms = df_with_rationale['NT_Mechanism'].unique()
    mechanisms_with_signal = [m for m in mechanisms if m != 'UNCERTAIN_SIGNIFICANCE']
    
    print(f"\nSampling from {len(mechanisms_with_signal)} mechanisms with signal...")
    
    for mech in mechanisms_with_signal:
        mech_df = df_with_rationale[df_with_rationale['NT_Mechanism'] == mech]
        n_available = len(mech_df)
        n_sample = min(n_per_category, n_available)
        
        if n_sample > 0:
            sampled = mech_df.sample(n=n_sample, random_state=random_state)
            samples.append(sampled)
            print(f"  {mech}: sampled {n_sample}/{n_available}")
    
    # Also sample some UNCERTAIN_SIGNIFICANCE (both pathogenic and benign)
    uncertain = df_with_rationale[df_with_rationale['NT_Mechanism'] == 'UNCERTAIN_SIGNIFICANCE']
    if len(uncertain) > 0:
        # Try to get mix of pathogenic and benign
        uncertain_path = uncertain[uncertain['label'].str.contains('athogenic', case=False, na=False)]
        uncertain_benign = uncertain[uncertain['label'].str.contains('enign', case=False, na=False)]
        
        n_path = min(n_per_category, len(uncertain_path))
        n_benign = min(n_per_category, len(uncertain_benign))
        
        if n_path > 0:
            samples.append(uncertain_path.sample(n=n_path, random_state=random_state))
            print(f"  UNCERTAIN_SIGNIFICANCE (Pathogenic): sampled {n_path}")
        if n_benign > 0:
            samples.append(uncertain_benign.sample(n=n_benign, random_state=random_state))
            print(f"  UNCERTAIN_SIGNIFICANCE (Benign): sampled {n_benign}")
    
    # Combine
    if samples:
        sample_df = pd.concat(samples, ignore_index=True)
    else:
        sample_df = pd.DataFrame()
    
    print(f"\nTotal sampled: {len(sample_df)}")
    
    return sample_df


def format_for_llm(sample_df):
    """
    Format sampled variants for pasting into chat.
    Returns a string ready to copy-paste.
    """
    output_lines = []
    
    output_lines.append("=" * 80)
    output_lines.append("VARIANTS FOR LLM EVALUATION")
    output_lines.append("=" * 80)
    output_lines.append("")
    
    for idx, row in sample_df.iterrows():
        var_id = row.get('#VariationID', row.get('VariationID', f'Row_{idx}'))
        
        output_lines.append(f"### VARIANT {idx + 1}: {var_id}")
        output_lines.append("-" * 40)
        
        # ClinVar info
        output_lines.append(f"**Label:** {row.get('label', 'N/A')}")
        output_lines.append(f"**VEP Consequence:** {row.get('sognlab_consequence', 'N/A')}")
        output_lines.append(f"**ClinVar Rationale:**")
        rationale = row.get('FullRationale', 'N/A')
        # Truncate very long rationales
        if len(str(rationale)) > 1000:
            rationale = str(rationale)[:1000] + "... [truncated]"
        output_lines.append(f"{rationale}")
        output_lines.append("")
        
        # NT Prediction
        output_lines.append(f"**NT Mechanism:** {row.get('NT_Mechanism', 'N/A')}")
        output_lines.append(f"**NT Confidence:** {row.get('NT_Confidence', 'N/A')}")
        output_lines.append(f"**NT Description:** {row.get('NT_Description', 'N/A')}")
        output_lines.append(f"**NT Trigger:** {row.get('NT_Trigger', 'N/A')}")
        output_lines.append(f"**All Mechanisms:** {row.get('NT_All_Mechanisms', 'N/A')}")
        output_lines.append("")
        
        # Top signals
        output_lines.append(f"**Top Signals (by magnitude):** {row.get('Top_Abs_Signals', 'N/A')}")
        output_lines.append(f"**Top Gains:** {row.get('Top_Gain_Signals', 'N/A')}")
        output_lines.append(f"**Top Losses:** {row.get('Top_Loss_Signals', 'N/A')}")
        output_lines.append("")
        output_lines.append("=" * 80)
        output_lines.append("")
    
    return "\n".join(output_lines)


# =============================================================================
# 4. MAIN EXECUTION
# =============================================================================

def main():
    # --- CONFIGURATION ---
    INPUT_FILE = "all_variants_with_rationales.csv"
    OUTPUT_FILE = "variants_with_mechanisms.csv"
    SAMPLE_OUTPUT_FILE = "sample_for_llm.txt"
    N_PER_CATEGORY = 2  # Variants per mechanism category
    
    # --- LOAD DATA ---
    print("Loading data...")
    df = pd.read_csv(INPUT_FILE, index_col=0)
    print(f"Loaded {len(df)} variants")
    
    # Show columns
    print(f"\nColumns: {df.columns.tolist()}")
    
    # Identify delta columns
    delta_cols = [c for c in df.columns if c.startswith("D_BED_")]
    print(f"Found {len(delta_cols)} delta columns")
    
    # Ensure numeric
    df[delta_cols] = df[delta_cols].apply(pd.to_numeric, errors='coerce')
    
    # --- ASSIGN MECHANISMS ---
    print("\nAssigning NT mechanisms...")
    mechanism_results = df.apply(
        lambda row: assign_mechanisms(row, NT_HYPOTHESIS_LOGIC),
        axis=1,
        result_type='expand'
    )
    for col in mechanism_results.columns:
        df[col] = mechanism_results[col]
    
    # --- EXTRACT TOP SIGNALS ---
    print("Extracting top signals...")
    signal_results = df.apply(
        lambda row: extract_top_signals(row, delta_cols),
        axis=1,
        result_type='expand'
    )
    for col in signal_results.columns:
        df[col] = signal_results[col]
    
    # --- SUMMARY ---
    print(f"\n{'='*60}")
    print("MECHANISM ASSIGNMENT SUMMARY")
    print(f"{'='*60}")
    print(df['NT_Mechanism'].value_counts())
    
    print(f"\nConfidence distribution:")
    print(df['NT_Confidence'].value_counts())
    
    # --- SAVE FULL RESULTS ---
    df.to_csv(OUTPUT_FILE)
    print(f"\nFull results saved to: {OUTPUT_FILE}")
    
    # --- SAMPLE FOR LLM ---
    print(f"\n{'='*60}")
    print("SAMPLING FOR LLM EVALUATION")
    print(f"{'='*60}")
    
    sample_df = sample_for_llm_evaluation(df, n_per_category=N_PER_CATEGORY)
    
    if len(sample_df) > 0:
        # Format for pasting
        llm_text = format_for_llm(sample_df)
        
        # Save to file
        with open(SAMPLE_OUTPUT_FILE, 'w') as f:
            f.write(llm_text)
        print(f"\nSample saved to: {SAMPLE_OUTPUT_FILE}")
        
        # Also print to console
        print(f"\n{'='*60}")
        print("COPY-PASTE THE FOLLOWING INTO CHAT:")
        print(f"{'='*60}")
        print(llm_text)
    else:
        print("ERROR: No variants with rationales found for sampling!")
        print("Check that 'FullRationale' column exists and has content.")
    
    return df, sample_df


if __name__ == "__main__":
    df, sample_df = main()

Loading data...
Loaded 41215 variants

Columns: ['chrom', 'pos', 'ref', 'alt', 'label', 'GeneSymbol', 'FullRationale', '#VariationID', 'sognlab_consequence', 'D_BED_protein_coding_gene', 'D_BED_lncRNA', 'D_BED_exon', 'D_BED_intron', 'D_BED_splice_donor', 'D_BED_splice_acceptor', 'D_BED_CTCF-bound', 'D_BED_polyA_signal', 'D_BED_enhancer_Tissue_specific', 'D_BED_enhancer_Tissue_invariant', 'D_BED_promoter_Tissue_specific', 'D_BED_promoter_Tissue_invariant', 'D_BED_5UTR+', 'D_BED_5UTR-', 'D_BED_3UTR+', 'D_BED_3UTR-', 'D_BED_skipped_exon', 'D_BED_always_on_exon', 'D_BED_start_codon', 'D_BED_stop_codon', 'D_BED_ORF', 'region']
Found 21 delta columns

Assigning NT mechanisms...
Extracting top signals...

MECHANISM ASSIGNMENT SUMMARY
NT_Mechanism
UNCERTAIN_SIGNIFICANCE              39872
START_LOSS                            386
SPLICE_SITE_DESTROYED                 215
WEAK_EXON_CHANGE                      176
WEAK_SPLICE_CHANGE                    128
WEAK_REGULATORY_CHANGE                10

In [214]:
df["llm_prompt"].to_csv("llm_prompt.csv")

In [232]:
import pandas as pd

SYSTEM_PROMPT = """You are an expert molecular geneticist and variant classifier.

Your task is to evaluate whether a computational prediction of variant mechanism aligns with:
(A) the human-curated ClinVar rationale when available, OR
(B) the ClinVar label alone (Benign/Pathogenic) when rationale or #VariationID is missing.

You must use LLM biological judgment, not heuristic rules. Read and reason over the full ClinVar rationale text when present.


You will be given, per variant:
1) Variant identifiers: chrom, pos, ref, alt, #VariationID (may be missing), GeneSymbol
2) ClinVar label: Benign or Pathogenic
3) ClinVar rationale text (FullRationale) IF available; otherwise it may be empty or missing
4) Optional VEP consequence annotation (e.g., missense_variant, splice_donor_variant, etc.)
5) Nucleotide Transformer (NT_v3) prediction summary:
   - NT_Mechanism, NT_Confidence, NT_Description, NT_Trigger, NT_All_Mechanisms
   - Top_Abs_Signals, Top_Gain_Signals, Top_Loss_Signals (feature deltas: alt - ref)

IMPORTANT CONTEXT:
- NT predicts TRANSCRIPT-LEVEL feature changes only (splicing, exons/introns, UTRs, ORFs, promoters/enhancers/CTCF, polyA).
- NT does NOT model protein-level effects (amino-acid substitutions, domains, folding, enzymatic activity).
- Therefore:
  - For missense-driven pathogenic variants, it is expected that NT may show no strong transcript-level mechanism.
  - NT mechanisms are most informative for splice variants, start/stop loss, exon/ORF disruptions, and regulatory variants.

YOUR OUTPUT:
chrom,pos,ref,alt,#VariationID,GeneSymbol,
label,NT_Mechanism,NT_Confidence,
concordance,concordance_explanation,
mechanism_supported,rationale_mechanism,
nt_mechanism_appropriate,missed_by_nt,
novelty,confidence,evaluation_basis

Field definitions:
- label ∈ {Benign, Pathogenic}
- concordance ∈ {CONCORDANT, DISCORDANT, PARTIAL, NOT_APPLICABLE, UNCERTAIN}
- mechanism_supported ∈ {true, false, null}
  (Whether the NT-predicted transcript mechanism supports the ClinVar interpretation; null if not applicable)
- nt_mechanism_appropriate ∈ {true, false}
- missed_by_nt ∈ {true, false, null}
  (true if ClinVar implies a transcript-level mechanism that NT failed to detect; false otherwise; null if not applicable)
- novelty ∈ {true, false}
  true only if:
  - ClinVar rationale is absent or non-mechanistic AND
  - NT predicts a strong, confident transcript-level damaging mechanism
  - This represents a potentially novel mechanistic hypothesis
- confidence ∈ {HIGH, MEDIUM, LOW}
  (Confidence in the concordance evaluation itself)
- evaluation_basis ∈ {RATIONALE, LABEL_ONLY}

HOW TO JUDGE CONCORDANCE:
1) If a specific ClinVar rationale is present:
   - Read the FullRationale text carefully
   - Infer the biological mechanism implied by the rationale
   - Evaluate concordance between the NT mechanism and the rationale’s implied mechanism.
   - Set evaluation_basis=RATIONALE.

2) If ClinVar rationale is missing (e.g., #VariationID missing or FullRationale empty or “no rationale”):
   - Evaluate concordance between NT outputs and the ClinVar label alone.
   - Set evaluation_basis=LABEL_ONLY.
   - Interpretation rules:
     - Pathogenic label:
       - If NT predicts a strong transcript-disrupting mechanism (splice site destroyed, start loss, major exon or ORF loss), this supports pathogenicity and should be labeled CONCORDANT.
       - If NT is UNCERTAIN or weak and the consequence is missense, label NOT_APPLICABLE (protein-level effect is out of scope).
       - If NT is UNCERTAIN or weak and the consequence suggests splice, start, stop, frameshift, or regulatory disruption, label DISCORDANT (expected transcript signal missed).
     - Benign label:
       - If NT is UNCERTAIN or shows no strong damaging transcript mechanism, this supports benignity and should be labeled CONCORDANT.
       - If NT predicts a strong damaging transcript mechanism, this conflicts with benignity and should be labeled DISCORDANT.

CONCORDANCE DEFINITIONS:
- CONCORDANT: NT mechanism is consistent with the rationale (if present) or with the label (if rationale is missing).
- DISCORDANT: NT mechanism contradicts the rationale or label.
- PARTIAL: NT captures only part of the rationale or label implications (e.g., splice signal present but ClinVar emphasizes an additional protein-level mechanism).
- NOT_APPLICABLE: ClinVar interpretation is primarily protein-level and cannot be evaluated by NT, especially when NT is UNCERTAIN.
- UNCERTAIN: Insufficient information to make a confident judgment.

STYLE RULES (MANDATORY)
- Use expert biological reasoning, not heuristics
- Do not infer mechanisms from consequence labels alone
- Keep explanations brief and mechanism-focused
- Output nothing except the single CSV row
"""

def make_variant_prompt(row: pd.Series) -> str:
    """
    Build a per-variant USER prompt for LLM concordance judgment.
    Works whether #VariationID / FullRationale are present or missing.
    """
    # Safe getters
    def g(col, default=""):
        v = row.get(col, default)
        if pd.isna(v):
            return ""
        return str(v)

    chrom = g("chrom")
    pos = g("pos")
    ref = g("ref")
    alt = g("alt")
    varid = g("#VariationID")
    gene = g("GeneSymbol")

    label = g("label")
    consequence = g("sognlab_consequence")

    rationale = g("FullRationale")
    if rationale.strip() == "":
        rationale = "(MISSING)"

    nt_mech = g("NT_Mechanism")
    nt_conf = g("NT_Confidence")
    nt_desc = g("NT_Description")
    nt_trigger = g("NT_Trigger")
    nt_all = g("NT_All_Mechanisms")

    top_abs = g("Top_Abs_Signals")
    top_gain = g("Top_Gain_Signals")
    top_loss = g("Top_Loss_Signals")

    return f"""Variant identifiers:
chrom={chrom} pos={pos} ref={ref} alt={alt}
#VariationID={varid if varid.strip() else "(MISSING)"} 
GeneSymbol={gene}

ClinVar label: {label}
VEP consequence (optional): {consequence}

ClinVar rationale (may be missing):
{rationale}

NT_v3 prediction summary:
NT_Mechanism: {nt_mech}
NT_Confidence: {nt_conf}
NT_Description: {nt_desc}
NT_Trigger: {nt_trigger}
NT_All_Mechanisms: {nt_all}

Top NT delta signals (alt - ref):
Top_Abs_Signals: {top_abs}
Top_Gain_Signals: {top_gain}
Top_Loss_Signals: {top_loss}

"""

def build_prompts_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a new dataframe with an added 'llm_prompt' column.
    """
    out = df.copy()
    out["llm_prompt"] = out.apply(make_variant_prompt, axis=1)
    return out

df_prompt = build_prompts_table(df)


In [233]:
df_prompt

,chrom,pos,ref,alt,label,GeneSymbol,FullRationale,#VariationID,sognlab_consequence,D_BED_protein_coding_gene,...,NT_Mechanism,NT_Confidence,NT_Description,NT_Trigger,NT_All_Mechanisms,NT_Mechanism_Count,Top_Abs_Signals,Top_Gain_Signals,Top_Loss_Signals,llm_prompt
0,chr1,69511,A,G,Benign,NaN,No rationale provided.,NaN,NaN,0.000813,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,,,0,stop_codon=-0.011; always_on_exon=0.008; exon=0.007,None,stop_codon=-0.011,Variant identifiers:\nchrom=chr1 pos=69511 ref=A alt=G\n#VariationID=(MISSING) \nGeneSymbol=\n\n...
1,chr1,953279,T,C,Benign,NaN,No rationale provided.,NaN,NaN,0.002615,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,,,0,stop_codon=-0.013; splice_acceptor=0.008; polyA_signal=-0.008,None,stop_codon=-0.013,Variant identifiers:\nchrom=chr1 pos=953279 ref=T alt=C\n#VariationID=(MISSING) \nGeneSymbol=\n\...
2,chr1,973858,G,C,Benign,NaN,No rationale provided.,NaN,NaN,-0.004761,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,,,0,splice_donor=0.056; lncRNA=-0.017; start_codon=-0.014,splice_donor=0.056; skipped_exon=0.013; ORF=0.010,lncRNA=-0.017; start_codon=-0.014,Variant identifiers:\nchrom=chr1 pos=973858 ref=G alt=C\n#VariationID=(MISSING) \nGeneSymbol=\n\...
3,chr1,973929,T,C,Benign,NaN,No rationale provided.,NaN,NaN,-0.002591,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,,,0,intron=0.026; splice_donor=0.023; always_on_exon=-0.020,intron=0.026; splice_donor=0.023,always_on_exon=-0.020; exon=-0.018; polyA_signal=-0.013,Variant identifiers:\nchrom=chr1 pos=973929 ref=T alt=C\n#VariationID=(MISSING) \nGeneSymbol=\n\...
4,chr1,978953,C,G,Benign,NaN,No rationale provided.,NaN,NaN,0.000017,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,,,0,stop_codon=-0.006; enhancer_Tissue_specific=-0.006; always_on_exon=-0.005,None,None,Variant identifiers:\nchrom=chr1 pos=978953 ref=C alt=G\n#VariationID=(MISSING) \nGeneSymbol=\n\...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41210,chrY,2787426,C,G,Pathogenic,SRY,"DESC: Reported in one family with several affected females with 46,XY gonadal dysgenesis, includ...",9739.0,missense_variant,0.021540,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,,,0,enhancer_Tissue_specific=-0.065; ORF=0.063; exon=0.029,ORF=0.063; exon=0.029; always_on_exon=0.025,enhancer_Tissue_specific=-0.065,Variant identifiers:\nchrom=chrY pos=2787426 ref=C alt=G\n#VariationID=9739.0 \nGeneSymbol=SRY\n...
41211,chrY,2787515,C,A,Pathogenic,SRY,No detailed rationale provided.,492908.0,missense_variant,0.004168,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,,,0,exon=-0.141; always_on_exon=-0.130; ORF=-0.112,enhancer_Tissue_specific=0.046; stop_codon=0.020; 5UTR-=0.014,exon=-0.141; always_on_exon=-0.130; ORF=-0.112,Variant identifiers:\nchrom=chrY pos=2787515 ref=C alt=A\n#VariationID=492908.0 \nGeneSymbol=SRY...
41212,chrY,2787551,C,T,Pathogenic,SRY,No detailed rationale provided.,9754.0,missense_variant,-0.023064,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,,,0,promoter_Tissue_specific=0.045; start_codon=0.031; lncRNA=0.029,promoter_Tissue_specific=0.045; start_codon=0.031; lncRNA=0.029,protein_coding_gene=-0.023; 5UTR-=-0.011,Variant identifiers:\nchrom=chrY pos=2787551 ref=C alt=T\n#VariationID=9754.0 \nGeneSymbol=SRY\n...
41213,chrY,7063898,A,T,Pathogenic,TBL1Y,No detailed rationale provided.,625467.0,missense_variant,0.022077,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,,,0,lncRNA=-0.148; start_codon=-0.137; enhancer_Tissue_specific=0.110,enhancer_Tissue_specific=0.110; 5UTR+=0.023; protein_coding_gene=0.022,lncRNA=-0.148; start_codon=-0.137; splice_acceptor=-0.047,Variant identifiers:\nchrom=chrY pos=7063898 ref=A alt=T\n#VariationID=625467.0 \nGeneSymbol=TBL...


In [ ]:
import pandas as pd
import time
import os
import google.generativeai as genai
from google.api_core import exceptions

# --- CONFIGURATION ---
API_KEY = "REDACTED_API_KEY" # REPLACE WITH YOUR ACTUAL KEY
MODEL_NAME = "gemini-2.0-flash"
OUTPUT_FILE = "llm_results.csv"
# Delay between successful calls to stay under rate limits (4s = ~15 requests/min)
RATE_LIMIT_DELAY = 4 

# 1. SETUP: Define the System Prompt (Kept same as yours)
SYSTEM_PROMPT = """You are an expert molecular geneticist and variant classifier.
[... Your System Prompt Truncated for Brevity ...]
STYLE RULES (MANDATORY)
- Output nothing except the single CSV row
"""

# 2. HELPER: Robust API Call with Retry Logic
def call_llm_api_robust(user_content, system_content, model_name=MODEL_NAME, max_retries=5):
    """
    Calls the LLM with exponential backoff. 
    If it hits a rate limit, it waits and tries again rather than crashing.
    """
    # Configure API (Ensure this is done securely)
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel(model_name, system_instruction=system_content)

    for attempt in range(max_retries):
        try:
            # Generate content
            response = model.generate_content(user_content)
            return response.text.strip()
            
        except exceptions.ResourceExhausted:
            # HANDLE RATE LIMIT (429)
            wait_time = (2 ** attempt) * 5  # 5s, 10s, 20s, 40s...
            print(f"\n[Warning] Rate limit hit. Cooling down for {wait_time}s... (Attempt {attempt+1}/{max_retries})")
            time.sleep(wait_time)
            
        except Exception as e:
            # Handle other random errors (500s, malformed requests)
            print(f"\n[Error] Unexpected API Error: {e}")
            return None
            
    print(f"\n[Fail] Max retries exceeded for this prompt.")
    return None

# 3. MAIN FUNCTION
def process_variants(df, output_file=OUTPUT_FILE):
    # --- RESUME LOGIC ---
    # Check if file exists to determine where to start
    start_index = 0
    if os.path.exists(output_file):
        print(f"Found existing {output_file}. Checking progress...")
        try:
            # Read existing file to count processed rows
            # on_bad_lines='skip' ensures we don't crash on half-written lines
            existing_df = pd.read_csv(output_file, on_bad_lines='skip')
            
            # If your CSV has a unique ID, filtering by ID is safer. 
            # For now, we will simply skip the number of rows already written.
            processed_count = len(existing_df)
            start_index = processed_count
            print(f"Resuming from index {start_index}...")
        except pd.errors.EmptyDataError:
            print("File exists but is empty. Starting from scratch.")
    else:
        print(f"Creating new output file: {output_file}")
        header = "chrom,pos,ref,alt,#VariationID,GeneSymbol,label,NT_Mechanism,NT_Confidence,concordance,concordance_explanation,mechanism_supported,rationale_mechanism,nt_mechanism_appropriate,missed_by_nt,novelty,confidence,evaluation_basis"
        with open(output_file, "w") as f:
            f.write(header + "\n")

    # --- PROCESSING LOOP ---
    total_rows = len(df)
    
    # We slice the dataframe to start where we left off
    df_to_process = df.iloc[start_index:]
    
    print(f"Processing {len(df_to_process)} variants (Total dataset: {total_rows})...")

    for i, (index, row) in enumerate(df_to_process.iterrows()):
        current_step = start_index + i + 1
        variant_prompt = row['llm_prompt']
        
        if pd.isna(variant_prompt):
            continue

        print(f"Processing row {current_step}/{total_rows}...", end="\r")

        # Call API
        response_text = call_llm_api_robust(
            user_content=variant_prompt,
            system_content=SYSTEM_PROMPT
        )

        if response_text:
            cleaned_response = response_text.replace("```csv", "").replace("```", "").strip()
            
            # Save immediately
            with open(output_file, "a") as f:
                f.write(cleaned_response + "\n")
            
            # *** RATE LIMIT SLEEP ***
            # Crucial: Pause before the next request
            time.sleep(RATE_LIMIT_DELAY)
        else:
            print(f"\nSkipping Row {current_step} due to repeated API failures.")

    print(f"\nProcessing complete. Results in {output_file}")

# 4. EXECUTION
if __name__ == "__main__":
    # Ensure df_prompt is loaded from your previous steps
    # df_prompt = pd.read_csv("your_input.csv") 
    
    if 'df_prompt' in locals():
        process_variants(df_prompt)
    else:
        print("Error: 'df_prompt' dataframe is not defined.")

Creating new output file: llm_results.csv
Processing 41215 variants (Total dataset: 41215)...
Processing row 2/41215...
[Warning] Rate limit hit. Cooling down for 5s... (Attempt 1/5)
Processing row 3/41215...
[Warning] Rate limit hit. Cooling down for 5s... (Attempt 1/5)

[Warning] Rate limit hit. Cooling down for 10s... (Attempt 2/5)
Processing row 4/41215...
[Warning] Rate limit hit. Cooling down for 5s... (Attempt 1/5)
Processing row 12/41215...
[Warning] Rate limit hit. Cooling down for 5s... (Attempt 1/5)

[Warning] Rate limit hit. Cooling down for 10s... (Attempt 2/5)
Processing row 13/41215...
[Warning] Rate limit hit. Cooling down for 5s... (Attempt 1/5)

[Warning] Rate limit hit. Cooling down for 10s... (Attempt 2/5)
Processing row 16/41215...
[Warning] Rate limit hit. Cooling down for 5s... (Attempt 1/5)
Processing row 17/41215...
[Warning] Rate limit hit. Cooling down for 5s... (Attempt 1/5)

[Warning] Rate limit hit. Cooling down for 10s... (Attempt 2/5)

[Warning] Rate lim

In [ ]:
s = pd.read_csv("llm_results_1.csv")

,Unnamed: 0,chrom,pos,ref,alt,label,GeneSymbol,FullRationale,#VariationID,sognlab_consequence,...,NT_Mechanism,NT_Confidence,NT_Description,NT_Trigger,NT_All_Mechanisms,NT_Mechanism_Count,Top_Abs_Signals,Top_Gain_Signals,Top_Loss_Signals,llm_prompt
0,3031,chr1,201386394,T,C,Benign,LAD1,No rationale provided.,4343566.0,NaN,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,NaN,NaN,0,always_on_exon=-0.016; intron=0.007; CTCF-bound=-0.005,NaN,always_on_exon=-0.016,Variant identifiers:\nchrom=chr1 pos=201386394 ref=T alt=C\n#VariationID=4343566.0 \nGeneSymbol=...
1,40940,chrX,154532257,C,A,Pathogenic,G6PD,"DESC: At admission, the boy's height was 78cm, weight was 8kg, head circumference was 41cm, HGB ...",2504081.0,missense_variant,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,NaN,NaN,0,skipped_exon=-0.160; always_on_exon=0.154; intron=-0.036,always_on_exon=0.154; enhancer_Tissue_specific=0.025; enhancer_Tissue_invariant=0.017,skipped_exon=-0.160; intron=-0.036; start_codon=-0.031,Variant identifiers:\nchrom=chrX pos=154532257 ref=C alt=A\n#VariationID=2504081.0 \nGeneSymbol=...
2,11951,chr5,162149125,A,T,Pathogenic,GABRG2,DESC: This variant is not present in population databases (gnomAD no frequency). This missense c...,1516815.0,"intron_variant,missense_variant",...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,NaN,NaN,0,5UTR+=-0.023; ORF=0.012; skipped_exon=0.008,ORF=0.012,5UTR+=-0.023,Variant identifiers:\nchrom=chr5 pos=162149125 ref=A alt=T\n#VariationID=1516815.0 \nGeneSymbol=...
3,19610,chr10,93519749,A,T,Benign,CEP55,DESC: This variant is classified as benign based on ACMG/AMP sequence variant interpretation gui...,1209684.0,missense_variant,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,NaN,NaN,0,start_codon=-0.050; ORF=-0.009; lncRNA=0.007,NaN,start_codon=-0.050,Variant identifiers:\nchrom=chr10 pos=93519749 ref=A alt=T\n#VariationID=1209684.0 \nGeneSymbol=...
4,23359,chr12,40486608,C,G,Benign,NaN,No rationale provided.,NaN,NaN,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,NaN,NaN,0,CTCF-bound=-0.017; ORF=0.011; enhancer_Tissue_specific=-0.008,ORF=0.011,CTCF-bound=-0.017,Variant identifiers:\nchrom=chr12 pos=40486608 ref=C alt=G\n#VariationID=(MISSING) \nGeneSymbol=...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,6828,chr2,232409765,C,A,Benign,NaN,No rationale provided.,NaN,NaN,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,NaN,NaN,0,enhancer_Tissue_specific=-0.083; stop_codon=0.032; intron=0.022,stop_codon=0.032; intron=0.022; lncRNA=0.015,enhancer_Tissue_specific=-0.083,Variant identifiers:\nchrom=chr2 pos=232409765 ref=C alt=A\n#VariationID=(MISSING) \nGeneSymbol=...
296,33508,chr19,1475392,A,C,Benign,NaN,No rationale provided.,NaN,NaN,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,NaN,NaN,0,stop_codon=-0.071; 5UTR-=-0.042; polyA_signal=-0.030,ORF=0.022; 3UTR-=0.015; splice_donor=0.010,stop_codon=-0.071; 5UTR-=-0.042; polyA_signal=-0.030,Variant identifiers:\nchrom=chr19 pos=1475392 ref=A alt=C\n#VariationID=(MISSING) \nGeneSymbol=\...
297,8707,chr3,129481546,T,G,Pathogenic,IFT122,No detailed rationale provided.,4635.0,missense_variant,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,NaN,NaN,0,intron=0.026; skipped_exon=0.023; splice_donor=0.019,intron=0.026; skipped_exon=0.023; splice_donor=0.019,enhancer_Tissue_specific=-0.011; always_on_exon=-0.010,Variant identifiers:\nchrom=chr3 pos=129481546 ref=T alt=G\n#VariationID=4635.0 \nGeneSymbol=IFT...
298,10398,chr4,155928611,G,A,Benign,NaN,No rationale provided.,NaN,NaN,...,UNCERTAIN_SIGNIFICANCE,LOW,No strong NT signal detected,NaN,NaN,0,exon=0.040; skipped_exon=0.035; intron=-0.021,exon=0.040; skipped_exon=0.035; 5UTR-=0.017,intron=-0.021; ORF=-0.011,Variant identifiers:\nchrom=chr4 pos=155928611 ref=G alt=A\n#VariationID=(MISSING) \nGeneSymbol=...


In [ ]:
conc.rename(columns={0: "chrom", 1: "pos", 2: "ref", 3: "alt", 4: "rationale", 5: "#VariantID", 6: "Analysis"}, inplace=True)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,chr1,69511,A,G,(MISSING),NaN,CONCORDANT,"NT shows no strong damaging transcript-level mechanism, consistent with benign label.",True,NaN,True,False,MEDIUM,LABEL_ONLY
1,chr1,953279,T,C,(MISSING),NaN,CONCORDANT,"NT shows no strong damaging transcript-level mechanism, consistent with benign label.",True,NaN,True,False,MEDIUM,LABEL_ONLY
2,chr1,973858,G,C,(MISSING),NaN,CONCORDANT,"NT shows no strong damaging transcript-level mechanism, consistent with benign label.",True,NaN,True,False,MEDIUM,LABEL_ONLY
3,chr1,973929,T,C,(MISSING),NaN,CONCORDANT,"NT shows no strong damaging transcript-level mechanism, consistent with benign label.",True,NaN,True,False,MEDIUM,LABEL_ONLY
4,chr1,978953,C,G,(MISSING),NaN,CONCORDANT,"NT shows no strong damaging transcript-level mechanism, consistent with benign label.",True,NaN,True,False,MEDIUM,LABEL_ONLY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41210,chrY,2787426,C,G,9739.0,SRY,NOT_APPLICABLE,ClinVar rationale indicates protein-level missense mechanism; NT transcript-level signal is expe...,NaN,MISSENSE_PROTEIN,True,NaN,MEDIUM,RATIONALE
41211,chrY,2787515,C,A,492908.0,SRY,UNCERTAIN,ClinVar rationale does not specify a clear transcript-level mechanism to compare with NT.,NaN,UNSPECIFIED,True,NaN,LOW,RATIONALE
41212,chrY,2787551,C,T,9754.0,SRY,UNCERTAIN,ClinVar rationale does not specify a clear transcript-level mechanism to compare with NT.,NaN,UNSPECIFIED,True,NaN,LOW,RATIONALE
41213,chrY,7063898,A,T,625467.0,TBL1Y,UNCERTAIN,ClinVar rationale does not specify a clear transcript-level mechanism to compare with NT.,NaN,UNSPECIFIED,True,NaN,LOW,RATIONALE


claude llm judgement output

In [148]:
import pandas as pd

# Evaluation results
evaluations = [
    {
        "Variant_ID": "934462",
        "Label": "Pathogenic",
        "NT_Mechanism": "START_LOSS",
        "NT_Confidence": "HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar explicitly states 'affects the initiator methionine of the FANCA mRNA' and 'Disruption of the initiator codon'. NT correctly identified START_LOSS (start_codon=-0.449) with secondary ORF_DISRUPTION. Perfect mechanistic match."
    },
    {
        "Variant_ID": "372766",
        "Label": "Pathogenic",
        "NT_Mechanism": "START_LOSS",
        "NT_Confidence": "HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar describes 'Initiation codon variant' affecting 'the initiator methionine of the FAH mRNA'. NT correctly called START_LOSS (start_codon=-0.588). Exact mechanistic alignment."
    },
    {
        "Variant_ID": "7113",
        "Label": "Pathogenic",
        "NT_Mechanism": "WEAK_SPLICE_CHANGE",
        "NT_Confidence": "LOW",
        "Concordance": "PARTIAL",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states variant 'occurs in the last base pair of coding exon 12, which makes it likely to have some effect on normal mRNA splicing'. NT detected splice_donor=-0.490, just below the -0.5 threshold for SPLICE_SITE_DESTROYED. The mechanism is correct but undertriggered due to threshold. Consider lowering splice threshold to -0.45."
    },
    {
        "Variant_ID": "1072728",
        "Label": "Pathogenic",
        "NT_Mechanism": "WEAK_SPLICE_CHANGE",
        "NT_Confidence": "LOW",
        "Concordance": "PARTIAL",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states variant 'falls at the last nucleotide of exon 4, which is part of the consensus splice site' and 'may disrupt the consensus splice site'. NT detected splice_donor=-0.365, correctly identifying splice effect but below strong threshold. Appropriate LOW confidence call."
    },
    {
        "Variant_ID": "223219",
        "Label": "Pathogenic",
        "NT_Mechanism": "SPLICE_SITE_DESTROYED",
        "NT_Confidence": "HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar explicitly states 'RNA analysis indicates that this missense change induces altered splicing' and 'activation of a cryptic splice site'. NT correctly identified SPLICE_SITE_DESTROYED (splice_donor=-0.884). Perfect match - NT captured the primary splicing defect."
    },
    {
        "Variant_ID": "2152312",
        "Label": "Pathogenic",
        "NT_Mechanism": "SPLICE_SITE_DESTROYED",
        "NT_Confidence": "HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states variant 'falls at the last nucleotide of exon 17, which is part of the consensus splice site' and 'may disrupt the consensus splice site'. NT detected comprehensive cascade: SPLICE_SITE_DESTROYED, SPLICE_INDUCED_INTRON_RETENTION, CONSTITUTIVE_EXON_LOSS. Excellent mechanistic capture of splice disruption and downstream effects."
    },
    {
        "Variant_ID": "689403",
        "Label": "Pathogenic",
        "NT_Mechanism": "WEAK_REGULATORY_CHANGE",
        "NT_Confidence": "LOW",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "LOW",
        "Analysis": "No ClinVar rationale provided. NT detected weak regulatory signals (enhancer=0.309, CTCF=0.330). Cannot validate mechanism without rationale. The pathogenic label may be due to protein-level effects NT cannot detect."
    },
    {
        "Variant_ID": "374796",
        "Label": "Pathogenic",
        "NT_Mechanism": "WEAK_REGULATORY_CHANGE",
        "NT_Confidence": "LOW",
        "Concordance": "NOT_APPLICABLE",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar describes missense (p.Cys1483Arg) in MTOR kinase domain with functional studies showing 'significantly increase phosphorylation levels'. This is a protein-level gain-of-function effect NT cannot detect. NT's weak regulatory signal (enhancer=0.479) is likely incidental to the actual pathogenic mechanism."
    },
    {
        "Variant_ID": "4803",
        "Label": "Pathogenic",
        "NT_Mechanism": "WEAK_EXON_CHANGE",
        "NT_Confidence": "LOW",
        "Concordance": "NOT_APPLICABLE",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar describes missense (p.Cys133Trp) with functional studies showing 'this missense change affects SPTLC1 function'. This is a protein-level effect. NT detected weak exon/intron changes (intron=0.409, always_on_exon=-0.370) which may indicate incidental splicing effects or noise, but the primary pathogenic mechanism is protein dysfunction."
    },
    {
        "Variant_ID": "496789",
        "Label": "Pathogenic",
        "NT_Mechanism": "WEAK_EXON_CHANGE",
        "NT_Confidence": "LOW",
        "Concordance": "NOT_APPLICABLE",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar describes missense (p.Arg152Trp) with functional studies showing '<10% of normal activity in fibroblasts'. This is a protein-level enzymatic deficiency. NT's weak signals (always_on_exon=-0.270, skipped_exon=0.254) are incidental to the actual pathogenic mechanism which is amino acid substitution affecting enzyme function."
    },
    {
        "Variant_ID": "561994",
        "Label": "Pathogenic",
        "NT_Mechanism": "RECIPROCAL_EXON_LOSS_INTRON_GAIN",
        "NT_Confidence": "MEDIUM",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "No ClinVar rationale provided. NT detected coherent intronization pattern (exon=-0.414, intron=0.411, always_on_exon=-0.432). The reciprocal exon loss/intron gain is consistent with a splicing defect mechanism. Likely a true positive but cannot confirm without rationale."
    },
    {
        "Variant_ID": "425625",
        "Label": "Pathogenic",
        "NT_Mechanism": "RECIPROCAL_EXON_LOSS_INTRON_GAIN",
        "NT_Confidence": "MEDIUM",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'Experimental studies have shown that this variant disrupts mRNA splicing'. NT detected RECIPROCAL_EXON_LOSS_INTRON_GAIN (exon=-0.411, intron=0.403) with ORF loss (-0.353). This intronization pattern is exactly consistent with the documented splicing disruption."
    },
    {
        "Variant_ID": "1211",
        "Label": "Pathogenic",
        "NT_Mechanism": "CONSTITUTIVE_EXON_LOSS",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "PARTIAL",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "ClinVar describes missense (p.Gly336Arg) with 'In vitro functional studies show reduced activity'. However, NT detected massive splicing signals (exon=-0.839, ORF=-0.776, always_on_exon=-0.744, intron=0.717). This suggests the variant may have DUAL effects: both amino acid change AND splicing disruption. NT may be capturing an undocumented splicing effect."
    },
    {
        "Variant_ID": "21",
        "Label": "Pathogenic",
        "NT_Mechanism": "CONSTITUTIVE_EXON_LOSS",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "No ClinVar rationale provided. NT detected very strong signals: CONSTITUTIVE_EXON_LOSS, STRONG_CODING_LOSS, ORF_DISRUPTION, INTRON_RETENTION. The coherent pattern (intron=0.787, exon=-0.710, always_on_exon=-0.620, ORF=-0.534) strongly suggests severe splicing defect. Likely true positive."
    },
    {
        "Variant_ID": "9168",
        "Label": "Pathogenic",
        "NT_Mechanism": "WEAK_ORF_CHANGE",
        "NT_Confidence": "LOW",
        "Concordance": "NOT_APPLICABLE",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar describes SMN1 missense (p.Ala2Gly) with 'experimental evidence suggests this variant results in abnormal protein function'. This is protein-level dysfunction. NT's weak signals (5UTR+=0.389, ORF=-0.289) are incidental. The actual pathogenic mechanism is amino acid change affecting protein function."
    },
    {
        "Variant_ID": "1203928",
        "Label": "Pathogenic",
        "NT_Mechanism": "WEAK_ORF_CHANGE",
        "NT_Confidence": "LOW",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "LOW",
        "Analysis": "ClinVar provides minimal rationale: 'in silico analysis supports a deleterious effect'. NT detected ORF gain (0.357) and stop codon gain (0.148). Without detailed rationale, cannot determine if NT is detecting the actual mechanism or incidental effects."
    },
    {
        "Variant_ID": "39671",
        "Label": "Pathogenic",
        "NT_Mechanism": "INTRON_RETENTION",
        "NT_Confidence": "MEDIUM",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "No ClinVar rationale provided. NT detected coherent intron retention pattern (intron=0.609, exon=-0.387, always_on_exon=-0.446, splice_donor=-0.222). The signals suggest splicing defect with intron retention. Likely true positive mechanism."
    },
    {
        "Variant_ID": "99288",
        "Label": "Pathogenic",
        "NT_Mechanism": "INTRON_RETENTION",
        "NT_Confidence": "MEDIUM",
        "Concordance": "PARTIAL",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "ClinVar describes missense (p.Cys1490Tyr) with pathogenicity based on clinical observations and protein modeling. NT detected INTRON_RETENTION (intron=0.593, exon=-0.382). ClinVar doesn't mention splicing - NT may be detecting cryptic splicing effect not documented, or this could be incidental to the actual protein-level mechanism."
    },
    {
        "Variant_ID": "375946",
        "Label": "Pathogenic",
        "NT_Mechanism": "STRONG_CODING_LOSS",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "PARTIAL",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "ClinVar describes missense (p.Asp594Val) in BRAF with 'reported to affect BRAF protein function'. However, NT detected massive signals (ORF=-0.899, exon=-0.863, intron=0.830). This suggests potential DUAL mechanism: both amino acid change AND severe splicing disruption. The NT signals are too strong to be noise - this variant may have undocumented splicing effects."
    },
    {
        "Variant_ID": "959566",
        "Label": "Pathogenic",
        "NT_Mechanism": "STRONG_CODING_LOSS",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'Algorithms developed to predict the effect of sequence changes on RNA splicing suggest that this variant may create or strengthen a splice site'. NT detected STRONG_CODING_LOSS (exon=-0.906, intron=0.834, ORF=-0.817). NT correctly captured the predicted splicing effect with strong confidence."
    },
    {
        "Variant_ID": "1185262",
        "Label": "Benign",
        "NT_Mechanism": "ORF_DISRUPTION",
        "NT_Confidence": "MEDIUM",
        "Concordance": "INTERESTING_BENIGN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "This is BENIGN but NT detected ORF_DISRUPTION (ORF=-0.511) with exon/skipped_exon loss. No rationale provided. This case suggests: (1) ORF disruption at this location may be tolerated, (2) the signals may be noise, or (3) the variant is in a non-critical region. NT signals don't guarantee pathogenicity."
    },
    {
        "Variant_ID": "180463",
        "Label": "Pathogenic",
        "NT_Mechanism": "ORF_DISRUPTION",
        "NT_Confidence": "MEDIUM",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "No ClinVar rationale provided. NT detected ORF_DISRUPTION (ORF=-0.643) with exon loss signals (always_on_exon=-0.424, exon=-0.387). The pattern suggests coding/splicing disruption. Likely true positive but cannot confirm without rationale."
    },
    {
        "Variant_ID": "652314",
        "Label": "Pathogenic",
        "NT_Mechanism": "SPLICE_INDUCED_INTRON_RETENTION",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'Algorithms developed to predict the effect of sequence changes on RNA splicing suggest that this variant may disrupt the consensus splice site'. NT correctly identified SPLICE_INDUCED_INTRON_RETENTION (intron=0.567, splice_donor=-0.308). The compound mechanism captures both the splice weakening and its downstream consequence."
    },
    {
        "Variant_ID": "4429034",
        "Label": "Benign",
        "NT_Mechanism": "SPLICE_INDUCED_INTRON_RETENTION",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "INTERESTING_BENIGN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "This is BENIGN but NT detected massive signals: SPLICE_INDUCED_INTRON_RETENTION, CONSTITUTIVE_EXON_LOSS, STRONG_CODING_LOSS, ORF_DISRUPTION (exon=-0.704, intron=0.703, ORF=-0.643). This is a critical case - either the variant is in a non-essential gene/region, or NT is producing false positives here. Requires investigation."
    },
    {
        "Variant_ID": "619571",
        "Label": "Pathogenic",
        "NT_Mechanism": "CRYPTIC_ORF_GAIN",
        "NT_Confidence": "LOW_MEDIUM",
        "Concordance": "DISCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar clearly describes 'c.2634+2T>C intronic pathogenic mutation' at 'canonical splice donor site' expected to cause 'aberrant splicing'. NT detected CRYPTIC_ORF_GAIN (ORF=0.710, exon=0.364) - detecting downstream GAINS rather than the primary splice LOSS. NT missed the primary mechanism (splice destruction) and instead caught the consequence (exonization). Should have triggered SPLICE_SITE_DESTROYED."
    },
    {
        "Variant_ID": "3736",
        "Label": "Pathogenic",
        "NT_Mechanism": "CRYPTIC_ORF_GAIN",
        "NT_Confidence": "LOW_MEDIUM",
        "Concordance": "DISCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar describes c.313+1G>A in LDLR as a 'common cause of FH' affecting splice site. NT detected CRYPTIC_ORF_GAIN (ORF=0.503, exon=0.419, skipped_exon=0.418) - all GAINS. Similar to variant 619571, NT detected downstream exonization consequence but missed the primary splice site destruction. The intron loss (-0.033) is too weak."
    },
    {
        "Variant_ID": "13063",
        "Label": "Pathogenic",
        "NT_Mechanism": "START_LOSS_WITH_ORF_DISRUPTION",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "ClinVar provides minimal rationale (PM2_supporting, PVS1_supporting). NT detected START_LOSS_WITH_ORF_DISRUPTION (start_codon=-0.388, ORF=-0.432). The compound mechanism correctly captures both start codon loss and downstream ORF impact. PVS1 (null variant) is consistent with start loss."
    },
    {
        "Variant_ID": "488649",
        "Label": "Pathogenic",
        "NT_Mechanism": "START_LOSS_WITH_ORF_DISRUPTION",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'Disruption of the initiator codon' and 'This sequence change affects the initiator methionine of the IFT43 mRNA'. NT correctly identified START_LOSS_WITH_ORF_DISRUPTION (start_codon=-0.365, ORF=-0.316). Perfect mechanistic match for initiation codon disruption."
    },
    {
        "Variant_ID": "17660",
        "Label": "Pathogenic",
        "NT_Mechanism": "CRYPTIC_SPLICE_GAIN",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "PARTIAL",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "ClinVar describes missense (p.C64G) in BRCA1 RING domain with extensive clinical evidence. Pathogenicity attributed to amino acid change disrupting zinc binding. NT detected CRYPTIC_SPLICE_GAIN (splice_donor=0.554). ClinVar doesn't mention splicing - NT may be detecting an undocumented cryptic splice effect, or this could be incidental to the protein-level mechanism."
    },
    {
        "Variant_ID": "189982",
        "Label": "Pathogenic",
        "NT_Mechanism": "CRYPTIC_SPLICE_GAIN",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "No ClinVar rationale provided. NT detected CRYPTIC_SPLICE_GAIN (splice_donor=0.527) with secondary exon changes (always_on_exon=-0.280). The pattern suggests creation of cryptic splice site. Likely true positive mechanism but cannot confirm without rationale."
    },
    {
        "Variant_ID": "134520",
        "Label": "Benign",
        "NT_Mechanism": "EXON_SKIPPING_INDUCED",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "INTERESTING_BENIGN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "This is BENIGN but NT detected EXON_SKIPPING_INDUCED (skipped_exon=0.448, always_on_exon=-0.412). ClinVar notes it was 'assessed due to predicted null impact' but classified benign 'based on frequency'. This suggests exon skipping at this location may be tolerated. NT signals describe molecular effect, not pathogenicity."
    },
    {
        "Variant_ID": "1375572",
        "Label": "Pathogenic",
        "NT_Mechanism": "EXON_SKIPPING_INDUCED",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "PARTIAL",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "ClinVar describes missense (p.Q701R) in MLH1 with pathogenicity based on MSI-H tumors and conservation. NT detected EXON_SKIPPING_INDUCED (skipped_exon=0.441, always_on_exon=-0.556). ClinVar doesn't explicitly mention splicing, but the strong NT signals suggest potential undocumented splicing effect in addition to the amino acid change."
    },
    {
        "Variant_ID": "770699",
        "Label": "Benign",
        "NT_Mechanism": "STRONG_REGULATORY_GAIN",
        "NT_Confidence": "MEDIUM",
        "Concordance": "INTERESTING_BENIGN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "This is BENIGN with no rationale. NT detected STRONG_REGULATORY_GAIN (enhancer=0.639). This case demonstrates that regulatory element gains are not necessarily pathogenic - context matters. The variant may create an enhancer that doesn't affect critical gene expression."
    },
    {
        "Variant_ID": "6114",
        "Label": "Pathogenic",
        "NT_Mechanism": "STRONG_REGULATORY_GAIN",
        "NT_Confidence": "MEDIUM",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "LOW",
        "Analysis": "No ClinVar rationale provided. NT detected STRONG_REGULATORY_GAIN (enhancer=0.642) with secondary exon signals (always_on_exon=-0.288, skipped_exon=0.205). Without rationale, cannot determine if enhancer gain is the pathogenic mechanism. Regulatory gain causing pathogenicity is less common than loss."
    },
    {
        "Variant_ID": "813278",
        "Label": "Pathogenic",
        "NT_Mechanism": "SPLICE_INDUCED_EXONIZATION",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'Intronic variant directly or indirectly altering the +5 splice site' with 'splice predictors support a deleterious effect'. NT correctly identified SPLICE_INDUCED_EXONIZATION (exon=0.698, intron=-0.569, ORF=0.715). The compound mechanism perfectly captures splice-induced inclusion of intronic sequence as coding."
    },
    {
        "Variant_ID": "259996",
        "Label": "Benign",
        "NT_Mechanism": "SPLICE_INDUCED_EXONIZATION",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "INTERESTING_BENIGN",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar explicitly states variant is benign based on 'intact protein function, lack of segregation with disease'. NT detected SPLICE_INDUCED_EXONIZATION (exon=0.485, intron=-0.309, ORF=0.433). This case confirms NT signals describe molecular changes that may or may not be pathogenic depending on context. The exonization effect at this location is apparently tolerated."
    },
    {
        "Variant_ID": "1260874",
        "Label": "Benign",
        "NT_Mechanism": "ENHANCER_DESTRUCTION",
        "NT_Confidence": "MEDIUM",
        "Concordance": "INTERESTING_BENIGN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "This is BENIGN with no rationale. NT detected ENHANCER_DESTRUCTION (enhancer=-0.500). Demonstrates that enhancer loss at certain loci may be tolerated. Gene-specific and tissue-specific context determines pathogenicity of regulatory disruption."
    },
    {
        "Variant_ID": "1289145",
        "Label": "Benign",
        "NT_Mechanism": "ENHANCER_DESTRUCTION",
        "NT_Confidence": "MEDIUM",
        "Concordance": "INTERESTING_BENIGN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "This is BENIGN with no rationale. NT detected ENHANCER_DESTRUCTION (enhancer=-0.543). Similar to variant 1260874 - enhancer loss is tolerated at this location. The minimal other signals (promoter=-0.030, intron=-0.016) suggest this is an isolated regulatory effect without broader transcript impact."
    },
    {
        "Variant_ID": "99561",
        "Label": "Pathogenic",
        "NT_Mechanism": "START_LOSS_WITH_UTR_EXTENSION",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'Disruption of the initiator codon' and 'This sequence change affects the initiator methionine of the TYR mRNA'. NT correctly identified START_LOSS_WITH_UTR_EXTENSION (start_codon=-0.353, 5UTR+=0.399). The compound mechanism captures both the start loss and consequent 5'UTR extension."
    },
    {
        "Variant_ID": "1686021",
        "Label": "Pathogenic",
        "NT_Mechanism": "START_LOSS_WITH_UTR_EXTENSION",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "No ClinVar rationale provided. NT detected START_LOSS_WITH_UTR_EXTENSION (start_codon=-0.390, 5UTR+=0.328) with secondary ORF loss (-0.351). The coherent pattern strongly suggests initiation codon disruption. Likely true positive."
    },
    {
        "Variant_ID": "869316",
        "Label": "Pathogenic",
        "NT_Mechanism": "UTR5_DISRUPTION",
        "NT_Confidence": "LOW_MEDIUM",
        "Concordance": "PARTIAL",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'disrupts the translation initiation codon of the HBB mRNA and is predicted to interfere with HBB protein synthesis'. NT detected UTR5_DISRUPTION (5UTR-=0.443) but also captured start_codon=-0.324 and ORF=-0.408. Primary call should be START_LOSS_WITH_ORF_DISRUPTION, but the start_codon delta (-0.324) is below the -0.35 threshold. NT captured the mechanism in secondary signals but primary assignment is suboptimal."
    },
    {
        "Variant_ID": "1685900",
        "Label": "Pathogenic",
        "NT_Mechanism": "UTR5_DISRUPTION",
        "NT_Confidence": "LOW_MEDIUM",
        "Concordance": "PARTIAL",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'Disruption of the initiator codon' and 'This sequence change affects the initiator methionine of the KCNQ2 mRNA'. NT detected UTR5_DISRUPTION (5UTR-=0.403) but also captured start_codon=-0.383 and ORF=-0.280. Similar to variant 869316 - NT captured mechanism in signals but start_codon is below threshold. Consider lowering START_LOSS threshold slightly."
    },
    {
        "Variant_ID": "1271364",
        "Label": "Benign",
        "NT_Mechanism": "POLYA_SIGNAL_LOSS",
        "NT_Confidence": "LOW_MEDIUM",
        "Concordance": "INTERESTING_BENIGN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "This is BENIGN with no rationale. NT detected POLYA_SIGNAL_LOSS (polyA=-0.413). Demonstrates that polyA signal disruption at certain loci is tolerated. The minimal other signals (always_on_exon=-0.041, ORF=-0.035) suggest isolated effect."
    },
    {
        "Variant_ID": "29627",
        "Label": "Pathogenic",
        "NT_Mechanism": "STRONG_EXON_LOSS",
        "NT_Confidence": "MEDIUM",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "No ClinVar rationale provided. NT detected STRONG_EXON_LOSS (exon=-0.622, intron=0.611) with multiple secondary mechanisms (INTRON_RETENTION, RECIPROCAL_EXON_LOSS_INTRON_GAIN). The coherent pattern suggests splicing defect. Likely true positive."
    },
    {
        "Variant_ID": "430791",
        "Label": "Pathogenic",
        "NT_Mechanism": "STRONG_EXON_LOSS",
        "NT_Confidence": "MEDIUM",
        "Concordance": "UNCERTAIN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "No ClinVar rationale provided. NT detected STRONG_EXON_LOSS (exon=-0.575, intron=0.535, always_on_exon=-0.279). The reciprocal exon loss/intron gain pattern is consistent with splicing defect. Likely true positive."
    },
    {
        "Variant_ID": "1259461",
        "Label": "Benign",
        "NT_Mechanism": "CRYPTIC_EXON_INCLUSION",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "INTERESTING_BENIGN",
        "Eval_Confidence": "MEDIUM",
        "Analysis": "This is BENIGN with no rationale. NT detected CRYPTIC_EXON_INCLUSION (skipped_exon=0.541, exon=0.556). Demonstrates that cryptic exon inclusion at certain locations may be tolerated. The 5UTR+ gain (0.408) suggests this may affect non-coding regions."
    },
    {
        "Variant_ID": "372440",
        "Label": "Pathogenic",
        "NT_Mechanism": "CRYPTIC_EXON_INCLUSION",
        "NT_Confidence": "MEDIUM_HIGH",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'Canonical splice site variant predicted to result in a null allele'. NT detected CRYPTIC_EXON_INCLUSION (skipped_exon=0.546, exon=0.590). The exon/ORF gains suggest aberrant inclusion of intronic sequence as consequence of splice site disruption. NT captured the downstream effect correctly."
    },
    {
        "Variant_ID": "1806955",
        "Label": "Pathogenic",
        "NT_Mechanism": "RECIPROCAL_EXON_GAIN_INTRON_LOSS",
        "NT_Confidence": "MEDIUM",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'expected to severely impact normal RNA splicing' and 'disruption of this splice site results in activation of a cryptic splice site and introduces a new termination codon'. NT detected RECIPROCAL_EXON_GAIN_INTRON_LOSS (exon=0.439, intron=-0.594). The exonization pattern matches the documented cryptic splice activation mechanism."
    },
    {
        "Variant_ID": "240888",
        "Label": "Pathogenic",
        "NT_Mechanism": "RECIPROCAL_EXON_GAIN_INTRON_LOSS",
        "NT_Confidence": "MEDIUM",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states 'aberrant splicing of intron 4' resulting in 'in-frame addition of 33 [amino acids]'. NT detected RECIPROCAL_EXON_GAIN_INTRON_LOSS (exon=0.455, intron=-0.767). The strong intron loss with exon gain perfectly captures the documented exonization mechanism."
    },
    {
        "Variant_ID": "3255266",
        "Label": "Benign",
        "NT_Mechanism": "CTCF_BOUNDARY_DISRUPTION",
        "NT_Confidence": "LOW_MEDIUM",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states variant is benign 'based on local population frequency' (57% in cohort). NT detected CTCF_BOUNDARY_DISRUPTION (CTCF=-0.469). The high population frequency confirms CTCF disruption at this location is tolerated. Appropriate that this is benign despite NT signal."
    },
    {
        "Variant_ID": "3060186",
        "Label": "Benign",
        "NT_Mechanism": "CTCF_BOUNDARY_DISRUPTION",
        "NT_Confidence": "LOW_MEDIUM",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states benign based on ACMG/AMP guidelines. NT detected CTCF_BOUNDARY_DISRUPTION (CTCF=-0.464) with minimal other signals. The isolated CTCF signal in a benign variant confirms context-dependent pathogenicity of chromatin boundary disruption."
    },
    {
        "Variant_ID": "178916",
        "Label": "Pathogenic",
        "NT_Mechanism": "ABERRANT_EXON_INCLUSION",
        "NT_Confidence": "MEDIUM",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states '396+1G>A variant in EDA...occurs in the invariant region (+/-1,2) of the splice consensus sequence and is predicted to cause altered splicing'. NT detected ABERRANT_EXON_INCLUSION (always_on_exon=0.774, intron=-0.753). The strong intron loss with exon/always_on_exon gain captures the splice-induced aberrant inclusion mechanism."
    },
    {
        "Variant_ID": "438831",
        "Label": "Pathogenic",
        "NT_Mechanism": "UNCERTAIN_SIGNIFICANCE",
        "NT_Confidence": "LOW",
        "Concordance": "NOT_APPLICABLE",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar describes 'reduced transcript amount and decreased MTSO1 protein based on RT-PCR and western blotting'. This suggests expression-level effect. NT correctly called UNCERTAIN_SIGNIFICANCE with minimal signals (max=0.022). NT cannot detect expression level changes from sequence alone - appropriate that no mechanism was called."
    },
    {
        "Variant_ID": "16122",
        "Label": "Pathogenic",
        "NT_Mechanism": "UNCERTAIN_SIGNIFICANCE",
        "NT_Confidence": "LOW",
        "Concordance": "NOT_APPLICABLE",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar describes missense (p.Ser498Leu) in GLUD1 with 'functional studies in HEK293T cells have shown that this variant leads to a gain in enzyme function'. This is a protein-level gain-of-function effect. NT correctly called UNCERTAIN_SIGNIFICANCE with minimal signals (max=0.031). NT cannot detect enzyme activity changes."
    },
    {
        "Variant_ID": "3059492",
        "Label": "Benign",
        "NT_Mechanism": "UNCERTAIN_SIGNIFICANCE",
        "NT_Confidence": "LOW",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states benign based on ACMG/AMP guidelines. NT correctly called UNCERTAIN_SIGNIFICANCE with essentially no signals (max=0.020). Benign variant with no molecular effect detected - perfect negative case."
    },
    {
        "Variant_ID": "1279127",
        "Label": "Benign",
        "NT_Mechanism": "UNCERTAIN_SIGNIFICANCE",
        "NT_Confidence": "LOW",
        "Concordance": "CONCORDANT",
        "Eval_Confidence": "HIGH",
        "Analysis": "ClinVar states benign based on ACMG/AMP guidelines. NT correctly called UNCERTAIN_SIGNIFICANCE with minimal signals (max=0.028). Benign variant with no significant molecular effect - appropriate negative call."
    }
]

# Create DataFrame
eval_df = pd.DataFrame(evaluations)

# Reorder columns
eval_df = eval_df[['Variant_ID', 'Label', 'NT_Mechanism', 'NT_Confidence', 'Concordance', 'Eval_Confidence', 'Analysis']]

# Save to CSV
eval_df.to_csv("llm_evaluation_results.csv", index=False)

# Print summary statistics
print("="*80)
print("EVALUATION SUMMARY")
print("="*80)

print("\n--- Concordance Distribution ---")
print(eval_df['Concordance'].value_counts())

print("\n--- Concordance by NT Confidence ---")
print(pd.crosstab(eval_df['NT_Confidence'], eval_df['Concordance']))

print("\n--- Concordance by Label ---")
print(pd.crosstab(eval_df['Label'], eval_df['Concordance']))

# Calculate key metrics
total = len(eval_df)
concordant = (eval_df['Concordance'] == 'CONCORDANT').sum()
partial = (eval_df['Concordance'] == 'PARTIAL').sum()
discordant = (eval_df['Concordance'] == 'DISCORDANT').sum()
uncertain = (eval_df['Concordance'] == 'UNCERTAIN').sum()
not_applicable = (eval_df['Concordance'] == 'NOT_APPLICABLE').sum()
interesting_benign = (eval_df['Concordance'] == 'INTERESTING_BENIGN').sum()

print("\n--- Summary Metrics ---")
print(f"Total variants evaluated: {total}")
print(f"CONCORDANT: {concordant} ({concordant/total*100:.1f}%)")
print(f"PARTIAL: {partial} ({partial/total*100:.1f}%)")
print(f"DISCORDANT: {discordant} ({discordant/total*100:.1f}%)")
print(f"UNCERTAIN (no rationale): {uncertain} ({uncertain/total*100:.1f}%)")
print(f"NOT_APPLICABLE (protein-level): {not_applicable} ({not_applicable/total*100:.1f}%)")
print(f"INTERESTING_BENIGN (signal in benign): {interesting_benign} ({interesting_benign/total*100:.1f}%)")

# Effective concordance (excluding uncertain and not_applicable)
evaluable = total - uncertain - not_applicable
if evaluable > 0:
    effective_concordance = (concordant + partial) / evaluable
    print(f"\nEffective concordance rate (CONCORDANT+PARTIAL / evaluable): {effective_concordance*100:.1f}%")
    strict_concordance = concordant / evaluable
    print(f"Strict concordance rate (CONCORDANT only / evaluable): {strict_concordance*100:.1f}%")

print("\n" + "="*80)
print("Results saved to: llm_evaluation_results.csv")
print("="*80)

EVALUATION SUMMARY

--- Concordance Distribution ---
Concordance
CONCORDANT            19
UNCERTAIN             11
PARTIAL                9
INTERESTING_BENIGN     9
NOT_APPLICABLE         6
DISCORDANT             2
Name: count, dtype: int64

--- Concordance by NT Confidence ---
Concordance    CONCORDANT  DISCORDANT  INTERESTING_BENIGN  NOT_APPLICABLE  \
NT_Confidence                                                               
HIGH                    4           0                   0               0   
LOW                     2           0                   0               6   
LOW_MEDIUM              2           2                   1               0   
MEDIUM                  4           0                   4               0   
MEDIUM_HIGH             7           0                   4               0   

Concordance    PARTIAL  UNCERTAIN  
NT_Confidence                      
HIGH                 0          0  
LOW                  2          2  
LOW_MEDIUM           2          0  
M

In [42]:
# --- 3. Load ClinVar variant_summary (Optimized) ---
variant_summary_path = "variant_summary.txt.gz"

# Only load columns needed for the mapping to save memory
clinvar_cols = ["Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF", "VariationID", "Type"]

var_df_summary = pd.read_csv(
    variant_summary_path,
    sep="\t",
    compression="gzip",
    # usecols=clinvar_cols, 
    low_memory=False
)


Claude API judgement

In [ ]:
import pandas as pd
import numpy as np
import json
import time

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

ANTHROPIC_API_KEY = "your-api-key-here"  # Or use environment variable
MODEL = "claude-sonnet-4-20250514"
MAX_TOKENS = 1024
BATCH_SIZE = 50  # Process in batches to avoid rate limits
SLEEP_BETWEEN_CALLS = 0.5  # Seconds between API calls

# =============================================================================
# 2. NT HYPOTHESIS LOGIC (Combined from your files)
# =============================================================================

NT_HYPOTHESIS_LOGIC = {
    # --- TIER 0: DEFINITIVE LOSS-OF-FUNCTION ---
    "SPLICE_SITE_DESTROYED": {
        "primary": [
            {"feature": "D_BED_splice_donor", "threshold": -0.5, "direction": "negative"},
            {"feature": "D_BED_splice_acceptor", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "HIGH",
        "description": "Loss of canonical splice donor or acceptor site",
        "priority": 0
    },
    "START_LOSS": {
        "primary": [{"feature": "D_BED_start_codon", "threshold": -0.5, "direction": "negative"}],
        "secondary": [],
        "confidence": "HIGH",
        "description": "Loss of translation start codon",
        "priority": 0
    },
    "STOP_CODON_DISRUPTION": {
        "primary": [{"feature": "D_BED_stop_codon", "threshold": -0.5, "direction": "negative"}],
        "secondary": [],
        "confidence": "HIGH",
        "description": "Disruption of stop codon (read-through)",
        "priority": 0
    },

    # --- TIER 1: LIKELY FUNCTIONAL IMPACT ---
    "CRYPTIC_SPLICE_GAIN": {
        "primary": [
            {"feature": "D_BED_splice_donor", "threshold": 0.5, "direction": "positive"},
            {"feature": "D_BED_splice_acceptor", "threshold": 0.5, "direction": "positive"}
        ],
        "secondary": [],
        "confidence": "MEDIUM_HIGH",
        "description": "Creation of new cryptic splice site",
        "priority": 1
    },
    "EXON_SKIPPING_INDUCED": {
        "primary": [{"feature": "D_BED_skipped_exon", "threshold": 0.4, "direction": "positive"}],
        "secondary": [{"feature": "D_BED_always_on_exon", "threshold": -0.3, "direction": "negative"}],
        "confidence": "MEDIUM_HIGH",
        "description": "Induction of exon skipping",
        "priority": 1
    },
    "CONSTITUTIVE_EXON_LOSS": {
        "primary": [{"feature": "D_BED_always_on_exon", "threshold": -0.5, "direction": "negative"}],
        "secondary": [],
        "confidence": "MEDIUM_HIGH",
        "description": "Loss of constitutively included exon",
        "priority": 1
    },
    "ORF_DISRUPTION": {
        "primary": [{"feature": "D_BED_ORF", "threshold": -0.5, "direction": "negative"}],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Disruption of open reading frame",
        "priority": 1
    },
    "STRONG_EXON_LOSS": {
        "primary": [{"feature": "D_BED_exon", "threshold": -0.5, "direction": "negative"}],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Strong loss of exon identity",
        "priority": 1
    },
    "CRYPTIC_EXON_INCLUSION": {
        "primary": [{"feature": "D_BED_skipped_exon", "threshold": 0.5, "direction": "positive"}],
        "secondary": [{"feature": "D_BED_exon", "threshold": 0.4, "direction": "positive"}],
        "confidence": "MEDIUM_HIGH",
        "description": "Inclusion of cryptic/skipped exon into transcript",
        "priority": 1
    },
    "STRONG_CODING_LOSS": {
        "primary": [
            {"feature": "D_BED_exon", "threshold": -0.45, "direction": "negative"},
            {"feature": "D_BED_always_on_exon", "threshold": -0.45, "direction": "negative"}
        ],
        "secondary": [{"feature": "D_BED_ORF", "threshold": -0.3, "direction": "negative"}],
        "confidence": "MEDIUM_HIGH",
        "description": "Loss of coding exon identity with ORF impact",
        "priority": 1
    },

    # --- TIER 2: REGULATORY IMPACT ---
    "PROMOTER_DESTRUCTION": {
        "primary": [
            {"feature": "D_BED_promoter_Tissue_specific", "threshold": -0.5, "direction": "negative"},
            {"feature": "D_BED_promoter_Tissue_invariant", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Destruction of promoter element",
        "priority": 2
    },
    "ENHANCER_DESTRUCTION": {
        "primary": [
            {"feature": "D_BED_enhancer_Tissue_specific", "threshold": -0.5, "direction": "negative"},
            {"feature": "D_BED_enhancer_Tissue_invariant", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Destruction of enhancer element",
        "priority": 2
    },
    "STRONG_REGULATORY_GAIN": {
        "primary": [
            {"feature": "D_BED_enhancer_Tissue_specific", "threshold": 0.6, "direction": "positive"},
            {"feature": "D_BED_promoter_Tissue_specific", "threshold": 0.6, "direction": "positive"}
        ],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Creation of strong regulatory element",
        "priority": 2
    },
    "UTR5_DISRUPTION": {
        "primary": [
            {"feature": "D_BED_5UTR+", "threshold": -0.4, "direction": "negative"},
            {"feature": "D_BED_5UTR-", "threshold": 0.4, "direction": "positive"}
        ],
        "secondary": [],
        "confidence": "LOW_MEDIUM",
        "description": "Disruption of 5' UTR",
        "priority": 2
    },
    "UTR3_DISRUPTION": {
        "primary": [
            {"feature": "D_BED_3UTR+", "threshold": -0.4, "direction": "negative"},
            {"feature": "D_BED_3UTR-", "threshold": 0.4, "direction": "positive"}
        ],
        "secondary": [],
        "confidence": "LOW_MEDIUM",
        "description": "Disruption of 3' UTR",
        "priority": 2
    },
    "START_LOSS_UTR_EXTENSION": {
        "primary": [{"feature": "D_BED_start_codon", "threshold": -0.4, "direction": "negative"}],
        "secondary": [{"feature": "D_BED_5UTR+", "threshold": 0.4, "direction": "positive"}],
        "confidence": "MEDIUM",
        "description": "Start codon loss leading to 5'UTR extension",
        "priority": 2
    },
    "POLYA_SIGNAL_LOSS": {
        "primary": [{"feature": "D_BED_polyA_signal", "threshold": -0.4, "direction": "negative"}],
        "secondary": [],
        "confidence": "LOW_MEDIUM",
        "description": "Loss of polyadenylation signal",
        "priority": 2
    },
    "CTCF_BOUNDARY_DISRUPTION": {
        "primary": [{"feature": "D_BED_CTCF-bound", "threshold": -0.4, "direction": "negative"}],
        "secondary": [],
        "confidence": "LOW_MEDIUM",
        "description": "Disruption of CTCF binding / chromatin boundary",
        "priority": 2
    },

    # --- TIER 3: GAIN-OF-FUNCTION / COMPLEX ---
    "CRYPTIC_ORF_GAIN": {
        "primary": [{"feature": "D_BED_ORF", "threshold": 0.5, "direction": "positive"}],
        "secondary": [],
        "confidence": "LOW_MEDIUM",
        "description": "Creation of cryptic open reading frame",
        "priority": 3
    },
    "ABERRANT_EXON_INCLUSION": {
        "primary": [{"feature": "D_BED_always_on_exon", "threshold": 0.5, "direction": "positive"}],
        "secondary": [{"feature": "D_BED_intron", "threshold": -0.3, "direction": "negative"}],
        "confidence": "MEDIUM",
        "description": "Aberrant inclusion of intronic sequence as exon",
        "priority": 3
    },
    "INTRON_RETENTION": {
        "primary": [{"feature": "D_BED_intron", "threshold": 0.5, "direction": "positive"}],
        "secondary": [{"feature": "D_BED_exon", "threshold": -0.3, "direction": "negative"}],
        "confidence": "MEDIUM",
        "description": "Retention of intron in mature transcript",
        "priority": 3
    },
    "RECIPROCAL_EXON_GAIN_INTRON": {
        "primary": [{"feature": "D_BED_exon", "threshold": 0.4, "direction": "positive"}],
        "secondary": [{"feature": "D_BED_intron", "threshold": -0.4, "direction": "negative"}],
        "confidence": "MEDIUM",
        "description": "Exon gain with reciprocal intron loss (exonization)",
        "priority": 3
    },
    "RECIPROCAL_EXON_LOSS_INTRON": {
        "primary": [{"feature": "D_BED_exon", "threshold": -0.4, "direction": "negative"}],
        "secondary": [{"feature": "D_BED_intron", "threshold": 0.4, "direction": "positive"}],
        "confidence": "MEDIUM",
        "description": "Exon loss with reciprocal intron gain (intronization)",
        "priority": 3
    },
    "GENE_ARCHITECTURE_DISRUPTION": {
        "primary": [{"feature": "D_BED_protein_coding_gene", "threshold": -0.5, "direction": "negative"}],
        "secondary": [],
        "confidence": "MEDIUM",
        "description": "Disruption of overall gene architecture",
        "priority": 3
    },
    "LNCRNA_DISRUPTION": {
        "primary": [{"feature": "D_BED_lncRNA", "threshold": -0.5, "direction": "negative"}],
        "secondary": [],
        "confidence": "LOW",
        "description": "Disruption of lncRNA",
        "priority": 3
    },

    # --- TIER 4: WEAK SIGNALS ---
    "WEAK_SPLICE_CHANGE": {
        "primary": [
            {"feature": "D_BED_splice_donor", "threshold": 0.3, "direction": "any"},
            {"feature": "D_BED_splice_acceptor", "threshold": 0.3, "direction": "any"}
        ],
        "secondary": [],
        "confidence": "LOW",
        "description": "Weak change in splice site score",
        "priority": 4
    },
    "WEAK_REGULATORY_CHANGE": {
        "primary": [
            {"feature": "D_BED_enhancer_Tissue_specific", "threshold": 0.3, "direction": "any"},
            {"feature": "D_BED_promoter_Tissue_specific", "threshold": 0.3, "direction": "any"}
        ],
        "secondary": [],
        "confidence": "LOW",
        "description": "Weak change in regulatory element",
        "priority": 4
    },
}

CONFIDENCE_SCORES = {
    "HIGH": 1.0,
    "MEDIUM_HIGH": 0.8,
    "MEDIUM": 0.6,
    "LOW_MEDIUM": 0.4,
    "LOW": 0.2,
    "UNKNOWN": 0.1
}


# =============================================================================
# 3. MECHANISM ASSIGNMENT FUNCTIONS
# =============================================================================

def check_condition(value, threshold, direction):
    """Evaluate a single condition."""
    if pd.isna(value):
        value = 0
    if direction == "negative":
        return value <= threshold
    elif direction == "positive":
        return value >= threshold
    elif direction == "any":
        return abs(value) >= abs(threshold)
    return False

def assign_mechanism(row, hypothesis_logic, return_mode='all'):
    """
    Assign mechanisms with flexible output modes.
    
    Args:
        row: DataFrame row with delta features
        hypothesis_logic: Dict of mechanism definitions
        return_mode: 
            'first' - return only highest priority match
            'all' - return all matches
            'by_tier' - return best match per priority tier
    
    Logic:
        PRIMARY conditions: OR logic (any one must pass)
        SECONDARY conditions: AND logic (all must pass, if any defined)
    """
    sorted_hypotheses = sorted(hypothesis_logic.items(), key=lambda x: x[1]['priority'])
    
    all_matches = []
    
    for mech_name, rules in sorted_hypotheses:
        # =====================
        # PRIMARY CHECK (OR Logic)
        # =====================
        # At least ONE primary condition must be satisfied
        primary_match = False
        trigger_feature = None
        trigger_value = None
        
        for condition in rules['primary']:
            feature = condition['feature']
            threshold = condition['threshold']
            direction = condition['direction']
            
            val = row.get(feature, 0)
            if pd.isna(val):
                val = 0
            
            if check_condition(val, threshold, direction):
                primary_match = True
                trigger_feature = feature
                trigger_value = val
                break  # OR logic: stop at first match
        
        if not primary_match:
            continue  # Skip to next mechanism

        # =====================
        # SECONDARY CHECK (AND Logic)
        # =====================
        # ALL secondary conditions must be satisfied (if any exist)
        secondary_match = True
        secondary_triggers = []
        
        if rules.get('secondary'):  # Only check if secondary rules exist
            for condition in rules['secondary']:
                feature = condition['feature']
                threshold = condition['threshold']
                direction = condition['direction']
                
                val = row.get(feature, 0)
                if pd.isna(val):
                    val = 0
                
                if check_condition(val, threshold, direction):
                    secondary_triggers.append({
                        'feature': feature,
                        'value': val,
                        'threshold': threshold,
                        'direction': direction
                    })
                else:
                    secondary_match = False
                    break  # AND logic: fail on first miss
        
        # =====================
        # RECORD MATCH
        # =====================
        if secondary_match:
            match_info = {
                'mechanism': mech_name,
                'confidence': rules.get('confidence', 'UNKNOWN'),
                'confidence_score': CONFIDENCE_SCORES.get(rules.get('confidence', 'UNKNOWN'), 0.1),
                'description': rules.get('description', ''),
                'priority': rules['priority'],
                'primary_trigger': {
                    'feature': trigger_feature,
                    'value': trigger_value
                },
                'secondary_triggers': secondary_triggers,
                'has_secondary': len(rules.get('secondary', [])) > 0,
                'secondary_satisfied': len(secondary_triggers)
            }
            all_matches.append(match_info)
    
    # =====================
    # FORMAT OUTPUT
    # =====================
    if not all_matches:
        return {
            'NT_Mechanism': 'UNCERTAIN_SIGNIFICANCE',
            'NT_Confidence': 'LOW',
            'NT_Confidence_Score': 0.1,
            'NT_Description': 'No strong NT signal detected',
            'NT_Trigger': '',
            'NT_All_Mechanisms': '',
            'NT_Mechanism_Count': 0,
            'NT_Mechanisms_List': [],
            'NT_Primary_Priority': 99
        }
    
    # Sort by priority, then by confidence score
    all_matches.sort(key=lambda x: (x['priority'], -x['confidence_score']))
    
    # Primary mechanism (highest priority)
    primary = all_matches[0]
    
    # Format trigger string
    primary_trigger_str = f"{primary['primary_trigger']['feature'].replace('D_BED_', '')}={primary['primary_trigger']['value']:.3f}"
    if primary['secondary_triggers']:
        sec_str = "; ".join([
            f"{t['feature'].replace('D_BED_', '')}={t['value']:.3f}" 
            for t in primary['secondary_triggers']
        ])
        primary_trigger_str += f" [+{sec_str}]"
    
    # All mechanisms string
    all_mechs_str = "; ".join([
        f"{m['mechanism']}(P{m['priority']},{m['confidence']})" 
        for m in all_matches
    ])
    
    # Mechanisms by tier
    by_tier = {}
    for m in all_matches:
        tier = m['priority']
        if tier not in by_tier:
            by_tier[tier] = []
        by_tier[tier].append(m['mechanism'])
    
    return {
        'NT_Mechanism': primary['mechanism'],
        'NT_Confidence': primary['confidence'],
        'NT_Confidence_Score': primary['confidence_score'],
        'NT_Description': primary['description'],
        'NT_Trigger': primary_trigger_str,
        'NT_All_Mechanisms': all_mechs_str,
        'NT_Mechanism_Count': len(all_matches),
        'NT_Mechanisms_List': [m['mechanism'] for m in all_matches],
        'NT_Mechanisms_By_Tier': by_tier,
        'NT_Primary_Priority': primary['priority'],
        # Additional detail for LLM prompt
        'NT_Full_Matches': all_matches  # Full detail if needed
    }

def extract_top_signals(row, delta_cols, k=3):
    """Extract top K signals by magnitude, gain, and loss."""
    data = {}
    for c in delta_cols:
        val = row.get(c, 0)
        data[c] = val if not pd.isna(val) else 0
    
    # Top by absolute magnitude
    sorted_abs = sorted(data.items(), key=lambda x: abs(x[1]), reverse=True)[:k]
    abs_str = "; ".join([f"{key.replace('D_BED_', '')}={val:.3f}" for key, val in sorted_abs])
    
    # Top gains
    gains = [(k, v) for k, v in data.items() if v > 0.01]
    sorted_gains = sorted(gains, key=lambda x: x[1], reverse=True)[:k]
    gain_str = "; ".join([f"{key.replace('D_BED_', '')}={val:.3f}" for key, val in sorted_gains])
    
    # Top losses
    losses = [(k, v) for k, v in data.items() if v < -0.01]
    sorted_losses = sorted(losses, key=lambda x: x[1])[:k]
    loss_str = "; ".join([f"{key.replace('D_BED_', '')}={val:.3f}" for key, val in sorted_losses])
    
    return {
        'Top_Abs_Signals': abs_str,
        'Top_Gain_Signals': gain_str,
        'Top_Loss_Signals': loss_str
    }


# =============================================================================
# 4. LLM JUDGE PROMPTS
# =============================================================================

SYSTEM_PROMPT = """You are an expert molecular geneticist and variant classifier. Your task is to evaluate whether a computational prediction of variant mechanism aligns with the human-curated ClinVar rationale.

You will be given:
1. A variant's ClinVar rationale (human-written explanation)
2. The ClinVar pathogenicity label
3. A predicted mechanism from a neural network model (Nucleotide Transformer)
4. The top genomic feature changes detected by the model
5. Optionally, the VEP consequence annotation

Your job is to assess the CONCORDANCE between the predicted mechanism and the ClinVar rationale.

IMPORTANT CONTEXT:
- The Nucleotide Transformer (NT) predicts changes in TRANSCRIPT-LEVEL features (splice sites, exons, introns, UTRs, ORFs, regulatory elements)
- NT CANNOT predict protein-level effects (amino acid changes, protein stability, domain disruption)
- For missense variants, NT might detect NO mechanism even if the variant is pathogenic - this is EXPECTED, not an error
- NT mechanisms are most informative for: splice variants, frameshift, start/stop loss, regulatory variants

Respond with a JSON object containing:
{
    "concordance": "CONCORDANT" | "DISCORDANT" | "PARTIAL" | "NOT_APPLICABLE" | "UNCERTAIN",
    "concordance_explanation": "Brief explanation of your judgment",
    "mechanism_supported": true | false | null,
    "rationale_mechanism": "What mechanism does the ClinVar rationale describe?",
    "nt_mechanism_appropriate": true | false,
    "missed_by_nt": "If NT missed something it should have caught, what?",
    "confidence": "HIGH" | "MEDIUM" | "LOW"
}

CONCORDANCE DEFINITIONS:
- CONCORDANT: NT mechanism matches or is consistent with ClinVar rationale
- DISCORDANT: NT mechanism contradicts or is incompatible with ClinVar rationale
- PARTIAL: NT captures part of the mechanism but misses other aspects
- NOT_APPLICABLE: ClinVar rationale describes protein-level effect that NT cannot detect (e.g., missense)
- UNCERTAIN: Insufficient information to judge"""


def build_variant_prompt(row):
    """Build the prompt for a single variant."""
    
    # Extract relevant fields
    variation_id = row.get('#VariationID', row.get('VariationID', 'Unknown'))
    clinvar_rationale = row.get('FullRationale', 'Not available')
    label = row.get('label', 'Unknown')
    consequence = row.get('sognlab_consequence', 'Not available')
    
    nt_mechanism = row.get('NT_Mechanism', 'UNCERTAIN_SIGNIFICANCE')
    nt_confidence = row.get('NT_Confidence', 'LOW')
    nt_description = row.get('NT_Description', '')
    nt_trigger = row.get('NT_Trigger', '')
    nt_all_mechanisms = row.get('NT_All_Mechanisms', '')

    
    top_abs = row.get('Top_Abs_Signals', '')
    top_gains = row.get('Top_Gain_Signals', '')
    top_losses = row.get('Top_Loss_Signals', '')
    
    prompt = f"""## Variant: {variation_id}

### ClinVar Information
**Pathogenicity Label:** {label}
**ClinVar Rationale:** 
{clinvar_rationale}

**VEP Consequence:** {consequence}

### Nucleotide Transformer (NT) Prediction
**Predicted Mechanism:** {nt_mechanism}
**Confidence:** {nt_confidence}
**Description:** {nt_description}
**Trigger Feature:** {nt_trigger}
**All Detected Mechanisms:** {nt_all_mechanisms}

### Top Feature Changes (Alt vs Ref)
**Strongest Changes:** {top_abs}
**Top Gains:** {top_gains}
**Top Losses:** {top_losses}

### Your Task
Evaluate the concordance between the NT prediction and the ClinVar rationale. Consider:
1. Does the NT mechanism match what ClinVar describes?
2. If NT says UNCERTAIN_SIGNIFICANCE, is this appropriate given the ClinVar rationale?
3. Is this a case where NT fundamentally cannot detect the mechanism (e.g., missense effect)?

Respond with the JSON object as specified."""

    return prompt


def build_batch_prompt(rows):
    """Build a prompt for multiple variants (batch processing)."""
    
    variants_text = []
    for idx, row in rows.iterrows():
        variant_prompt = build_variant_prompt(row)
        variants_text.append(f"---\n### VARIANT {idx}\n{variant_prompt}")
    
    batch_prompt = f"""Evaluate the following {len(rows)} variants. For EACH variant, provide a separate JSON object.

Return your response as a JSON array with one object per variant, in the same order as presented.

{chr(10).join(variants_text)}

---
Respond with a JSON array containing {len(rows)} objects, one for each variant above."""
    
    return batch_prompt


# =============================================================================
# 5. LLM API CALLS
# =============================================================================

def call_llm_single(client, prompt, system_prompt=SYSTEM_PROMPT):
    """Call LLM for a single variant."""
    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    except Exception as e:
        print(f"API Error: {e}")
        return None


def call_llm_batch(client, prompts, system_prompt=SYSTEM_PROMPT):
    """Call LLM for a batch of variants."""
    results = []
    for prompt in prompts:
        response = call_llm_single(client, prompt, system_prompt)
        results.append(response)
        time.sleep(SLEEP_BETWEEN_CALLS)
    return results


def parse_llm_response(response_text):
    """Parse LLM JSON response."""
    if response_text is None:
        return {
            "concordance": "ERROR",
            "concordance_explanation": "API call failed",
            "parse_error": True
        }
    
    try:
        # Try to extract JSON from response
        # Handle case where LLM wraps in markdown code blocks
        text = response_text.strip()
        if text.startswith("```json"):
            text = text[7:]
        if text.startswith("```"):
            text = text[3:]
        if text.endswith("```"):
            text = text[:-3]
        
        result = json.loads(text.strip())
        result["parse_error"] = False
        return result
    except json.JSONDecodeError as e:
        return {
            "concordance": "PARSE_ERROR",
            "concordance_explanation": f"Failed to parse: {str(e)[:100]}",
            "raw_response": response_text[:500],
            "parse_error": True
        }


# =============================================================================
# 6. MAIN PIPELINE
# =============================================================================

def run_llm_evaluation(df, client, sample_size=None, output_file="llm_evaluation_results.csv"):
    """
    Run LLM evaluation on variants.
    
    Args:
        df: DataFrame with variants (must have NT mechanisms assigned)
        client: Anthropic client
        sample_size: If set, evaluate only this many variants (for testing)
        output_file: Where to save results
    """
    
    # Filter to variants with rationales
    eval_df = df[df['FullRationale'].notna() & (df['FullRationale'] != '')].copy()
    print(f"Variants with ClinVar rationales: {len(eval_df)}")
    
    if sample_size:
        eval_df = eval_df.sample(n=min(sample_size, len(eval_df)), random_state=42)
        print(f"Sampled {len(eval_df)} variants for evaluation")
    
    # Process variants
    results = []
    
    for idx, (row_idx, row) in enumerate(eval_df.iterrows()):
        if idx % 10 == 0:
            print(f"Processing variant {idx+1}/{len(eval_df)}...")
        
        # Build prompt
        prompt = build_variant_prompt(row)
        
        # Call LLM
        response = call_llm_single(client, prompt)
        
        # Parse response
        parsed = parse_llm_response(response)
        
        # Store result
        result = {
            'original_index': row_idx,
            'VariationID': row.get('#VariationID', row.get('VariationID', '')),
            'NT_Mechanism': row.get('NT_Mechanism', ''),
            'NT_Confidence': row.get('NT_Confidence', ''),
            'label': row.get('label', ''),
            'sognlab_consequence': row.get('sognlab_consequence', ''),
            **parsed
        }
        results.append(result)
        
        # Rate limiting
        time.sleep(SLEEP_BETWEEN_CALLS)
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Save
    results_df.to_csv(output_file, index=False)
    print(f"\nResults saved to: {output_file}")
    
    # Print summary
    print_evaluation_summary(results_df)
    
    return results_df


def print_evaluation_summary(results_df):
    """Print summary of LLM evaluation results."""
    
    print(f"\n{'='*70}")
    print("LLM EVALUATION SUMMARY")
    print(f"{'='*70}")
    
    print(f"\nTotal variants evaluated: {len(results_df)}")
    
    # Concordance distribution
    print(f"\n--- Concordance Distribution ---")
    print(results_df['concordance'].value_counts())
    
    # By NT Mechanism
    print(f"\n--- Concordance by NT Mechanism ---")
    mech_concordance = results_df.groupby('NT_Mechanism')['concordance'].value_counts().unstack(fill_value=0)
    print(mech_concordance)
    
    # By label
    print(f"\n--- Concordance by Pathogenicity Label ---")
    label_concordance = results_df.groupby('label')['concordance'].value_counts().unstack(fill_value=0)
    print(label_concordance)
    
    # By consequence
    if 'sognlab_consequence' in results_df.columns:
        print(f"\n--- Concordance by VEP Consequence (top 10) ---")
        top_cons = results_df['sognlab_consequence'].value_counts().head(10).index
        cons_subset = results_df[results_df['sognlab_consequence'].isin(top_cons)]
        cons_concordance = cons_subset.groupby('sognlab_consequence')['concordance'].value_counts().unstack(fill_value=0)
        print(cons_concordance)
    
    # NT Mechanism appropriateness
    if 'nt_mechanism_appropriate' in results_df.columns:
        print(f"\n--- NT Mechanism Appropriateness ---")
        print(results_df['nt_mechanism_appropriate'].value_counts())
    
    # Calculate summary metrics
    total_evaluated = len(results_df[~results_df['concordance'].isin(['ERROR', 'PARSE_ERROR'])])
    if total_evaluated > 0:
        concordant = (results_df['concordance'] == 'CONCORDANT').sum()
        partial = (results_df['concordance'] == 'PARTIAL').sum()
        discordant = (results_df['concordance'] == 'DISCORDANT').sum()
        not_applicable = (results_df['concordance'] == 'NOT_APPLICABLE').sum()
        
        print(f"\n--- Summary Metrics ---")
        print(f"Concordant: {concordant} ({concordant/total_evaluated*100:.1f}%)")
        print(f"Partial: {partial} ({partial/total_evaluated*100:.1f}%)")
        print(f"Discordant: {discordant} ({discordant/total_evaluated*100:.1f}%)")
        print(f"Not Applicable (protein-level): {not_applicable} ({not_applicable/total_evaluated*100:.1f}%)")
        
        # Effective concordance (excluding NOT_APPLICABLE)
        applicable = total_evaluated - not_applicable
        if applicable > 0:
            effective_concordance = (concordant + partial) / applicable
            print(f"\nEffective concordance (excluding protein-level): {effective_concordance*100:.1f}%")


# =============================================================================
# 7. ANALYSIS OF DISCORDANT CASES
# =============================================================================

def analyze_discordant_cases(results_df, original_df):
    """Deep dive into discordant cases."""
    
    discordant = results_df[results_df['concordance'] == 'DISCORDANT'].copy()
    
    print(f"\n{'='*70}")
    print(f"ANALYSIS OF {len(discordant)} DISCORDANT CASES")
    print(f"{'='*70}")
    
    if len(discordant) == 0:
        print("No discordant cases found!")
        return
    
    # By mechanism
    print(f"\n--- Discordant by NT Mechanism ---")
    print(discordant['NT_Mechanism'].value_counts())
    
    # By consequence
    print(f"\n--- Discordant by VEP Consequence ---")
    print(discordant['sognlab_consequence'].value_counts().head(10))
    
    # Sample discordant cases
    print(f"\n--- Sample Discordant Cases ---")
    for idx, row in discordant.head(5).iterrows():
        print(f"\nVariant: {row['VariationID']}")
        print(f"  NT Mechanism: {row['NT_Mechanism']}")
        print(f"  Label: {row['label']}")
        print(f"  Consequence: {row['sognlab_consequence']}")
        print(f"  LLM Explanation: {row.get('concordance_explanation', 'N/A')[:200]}")
        print(f"  Rationale Mechanism: {row.get('rationale_mechanism', 'N/A')[:200]}")
    
    return discordant


# =============================================================================
# 8. MAIN EXECUTION
# =============================================================================

def main():
    # --- CONFIGURATION ---
    INPUT_FILE = "variants_with_rationales.csv"
    OUTPUT_MECHANISMS = "variants_with_nt_mechanisms.csv"
    OUTPUT_LLM_EVAL = "llm_evaluation_results.csv"
    SAMPLE_SIZE = 100  # Set to None for full evaluation (will take time/cost money)
    
    # --- LOAD DATA ---
    print("Loading data...")
    df = pd.read_csv(INPUT_FILE, index_col=0)
    print(f"Loaded {len(df)} variants")
    
    # Identify delta columns
    delta_cols = [c for c in df.columns if c.startswith("D_BED_")]
    print(f"Found {len(delta_cols)} delta columns")
    
    # Ensure numeric
    df[delta_cols] = df[delta_cols].apply(pd.to_numeric, errors='coerce')
    
    # --- ASSIGN NT MECHANISMS ---
    print("\nAssigning NT mechanisms...")
    mechanism_results = df.apply(
        lambda row: assign_mechanism(row, NT_HYPOTHESIS_LOGIC),
        axis=1,
        result_type='expand'
    )
    for col in mechanism_results.columns:
        df[col] = mechanism_results[col]
    
    # Extract top signals
    print("Extracting top signals...")
    signal_results = df.apply(
        lambda row: extract_top_signals(row, delta_cols),
        axis=1,
        result_type='expand'
    )
    for col in signal_results.columns:
        df[col] = signal_results[col]
    
    # Save intermediate results
    df.to_csv(OUTPUT_MECHANISMS)
    print(f"Mechanisms saved to: {OUTPUT_MECHANISMS}")
    
    # Print mechanism summary
    print(f"\n--- Mechanism Distribution ---")
    print(df['NT_Mechanism'].value_counts())
    
    # # --- LLM EVALUATION ---
    # print(f"\n{'='*70}")
    # print("STARTING LLM EVALUATION")
    # print(f"{'='*70}")
    
    # # Check for rationales
    # has_rationale = df['FullRationale'].notna() & (df['FullRationale'] != '')
    # print(f"Variants with ClinVar rationales: {has_rationale.sum()}")
    
    # if has_rationale.sum() == 0:
    #     print("ERROR: No variants have FullRationale column populated!")
    #     print("Available columns:", df.columns.tolist())
    #     return df, None
    
    # # Initialize Anthropic client
    # import os
    # api_key = os.environ.get('ANTHROPIC_API_KEY', ANTHROPIC_API_KEY)
    # if api_key == "your-api-key-here":
    #     print("\nWARNING: No API key provided. Set ANTHROPIC_API_KEY environment variable.")
    #     print("Skipping LLM evaluation. Returning mechanism assignments only.")
    #     return df, None
    
    # client = Anthropic(api_key=api_key)
    
    # # Run evaluation
    # results_df = run_llm_evaluation(
    #     df, 
    #     client, 
    #     sample_size=SAMPLE_SIZE,
    #     output_file=OUTPUT_LLM_EVAL
    # )
    
    # # Analyze discordant cases
    # if results_df is not None:
    #     discordant = analyze_discordant_cases(results_df, df)
    
    # return df, results_df


if __name__ == "__main__":
    df, results = main()

Index(['D_BED_protein_coding_gene', 'D_BED_lncRNA', 'D_BED_exon',
       'D_BED_intron', 'D_BED_splice_donor', 'D_BED_splice_acceptor',
       'D_BED_CTCF-bound', 'D_BED_polyA_signal',
       'D_BED_enhancer_Tissue_specific', 'D_BED_enhancer_Tissue_invariant',
       'D_BED_promoter_Tissue_specific', 'D_BED_promoter_Tissue_invariant',
       'D_BED_5UTR+', 'D_BED_5UTR-', 'D_BED_3UTR+', 'D_BED_3UTR-',
       'D_BED_skipped_exon', 'D_BED_always_on_exon', 'D_BED_start_codon',
       'D_BED_stop_codon', 'D_BED_ORF'],
      dtype='object')
21


older assignments

In [ ]:
import pandas as pd
import numpy as np

nt_hypothesis_logic = {
    # --- 1. CRITICAL STRUCTURAL CHANGES (Priority 1) ---
    "CANONICAL_SPLICE_LOSS": {
        "primary": [
            {"feature": "D_BED_splice_donor", "threshold": -0.5, "direction": "negative"},
            {"feature": "D_BED_splice_acceptor", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [], "priority": 1
    },
    
    "STRONG_SPLICE_SITE_GAIN": {
        "primary": [
            {"feature": "D_BED_splice_donor", "threshold": 0.6, "direction": "positive"},
            {"feature": "D_BED_splice_acceptor", "threshold": 0.6, "direction": "positive"}
        ],
        "secondary": [], "priority": 1
    },

    "START_CODON_LOSS": {
        "primary": [{"feature": "D_BED_start_codon", "threshold": -0.5, "direction": "negative"}],
        "secondary": [], "priority": 1
    },

    # --- 2. REGULATORY & UTR GAINS (New) ---
    
    # Covers 7706, 4855 (Enhancer > 0.6)
    "STRONG_REGULATORY_GAIN": {
        "description": "Creation of strong regulatory element (Enhancer/Promoter).",
        "primary": [
            {"feature": "D_BED_enhancer_Tissue_specific", "threshold": 0.6, "direction": "positive"},
            {"feature": "D_BED_promoter_Tissue_specific", "threshold": 0.6, "direction": "positive"}
        ],
        "secondary": [], 
        "priority": 2
    },
    
    # Covers 14541 (Start -0.45, 5UTR +0.5)
    "START_LOSS_UTR_EXTENSION": {
        "description": "Loss of Start Codon leading to 5'UTR extension.",
        "primary": [
             {"feature": "D_BED_start_codon", "threshold": -0.4, "direction": "negative"}
        ],
        "secondary": [
             {"feature": "D_BED_5UTR+", "threshold": 0.4, "direction": "positive"}
        ],
        "priority": 2
    },

    # --- 3. EXON/INTRON FLIPS (Priority 2) ---

    # Covers 14125, 14127 (Exon > 0.6, Skipped > 0.5)
    "CRYPTIC_EXON_INCLUSION": {
        "description": "Inclusion of a cryptic/skipped exon.",
        "primary": [
             {"feature": "D_BED_skipped_exon", "threshold": 0.5, "direction": "positive"}
        ],
        "secondary": [
             {"feature": "D_BED_exon", "threshold": 0.4, "direction": "positive"}
        ],
        "priority": 2
    },

    # Covers 12068, 10472, 14591 (Exon ~-0.5, AlwaysOn ~-0.5)
    # Loosened thresholds to -0.45
    "STRONG_CODING_LOSS": {
        "description": "Loss of coding exon identity (Exon Loss + AlwaysOn Loss).",
        "primary": [
             {"feature": "D_BED_exon", "threshold": -0.45, "direction": "negative"},
             {"feature": "D_BED_always_on_exon", "threshold": -0.45, "direction": "negative"}
        ],
        "secondary": [
             # Split into subtypes in logic engine if needed, or rely on primary strength
             {"feature": "D_BED_ORF", "threshold": -0.3, "direction": "negative"}
        ],
        "priority": 2
    },
    
    # Split for OR logic coverage on secondary features
    "STRONG_CODING_LOSS_SKIPPED": {
        "primary": [
             {"feature": "D_BED_exon", "threshold": -0.45, "direction": "negative"},
             {"feature": "D_BED_always_on_exon", "threshold": -0.45, "direction": "negative"}
        ],
        "secondary": [{"feature": "D_BED_skipped_exon", "threshold": -0.3, "direction": "negative"}],
        "priority": 2
    },

    # Existing robust gates
    "ABERRANT_EXON_INCLUSION": { 
        "primary": [{"feature": "D_BED_always_on_exon", "threshold": 0.5, "direction": "positive"}],
        "secondary": [{"feature": "D_BED_skipped_exon", "threshold": -0.5, "direction": "negative"}],
        "priority": 2
    },
    "RECIPROCAL_EXON_GAIN_INTRON": {
        "primary": [{"feature": "D_BED_exon", "threshold": 0.4, "direction": "positive"}],
        "secondary": [{"feature": "D_BED_intron", "threshold": -0.4, "direction": "negative"}],
        "priority": 2
    },
    "RECIPROCAL_EXON_LOSS_INTRON": {
        "primary": [{"feature": "D_BED_exon", "threshold": -0.4, "direction": "negative"}],
        "secondary": [{"feature": "D_BED_intron", "threshold": 0.4, "direction": "positive"}],
        "priority": 2
    },
    "CRYPTIC_TRANSLATION_GAIN": { 
        "primary": [{"feature": "D_BED_ORF", "threshold": 0.6, "direction": "positive"}],
        "secondary": [], "priority": 2
    },
    "STRONG_ORF_LOSS": { 
        "primary": [{"feature": "D_BED_ORF", "threshold": -0.6, "direction": "negative"}],
        "secondary": [], "priority": 2
    },
    "STRONG_REGULATORY_LOSS": { 
        "primary": [{"feature": "D_BED_enhancer_Tissue_specific", "threshold": -0.6, "direction": "negative"},
                    {"feature": "D_BED_promoter_Tissue_specific", "threshold": -0.6, "direction": "negative"}],
        "secondary": [], "priority": 2
    },
    "STRONG_INTRON_LOSS": { 
        "primary": [{"feature": "D_BED_intron", "threshold": -0.6, "direction": "negative"}],
        "secondary": [], "priority": 2
    }
}

# --- 2. LOGIC FUNCTIONS ---

def check_condition(value, threshold, direction):
    """Helper to evaluate different direction types."""
    if direction == "negative":
        return value <= threshold
    elif direction == "positive":
        return value >= threshold
    elif direction == "stable":
        return abs(value) < abs(threshold)
    elif direction == "any":
        return abs(value) >= abs(threshold)
    return False

def assign_mechanism(row, hypothesis_logic):
    """Evaluates NTv3 deltas against the hypothesis logic."""
    sorted_hypotheses = sorted(hypothesis_logic.items(), key=lambda x: x[1]['priority'])
    
    for mech_name, rules in sorted_hypotheses:
        # Check Primary (OR Logic)
        primary_match = False
        for condition in rules['primary']:
            val = row.get(condition['feature'], 0)
            if check_condition(val, condition['threshold'], condition['direction']):
                primary_match = True
                break
        
        if not primary_match: continue

        # Check Secondary (AND Logic)
        secondary_match = True
        for condition in rules['secondary']:
            val = row.get(condition['feature'], 0)
            if not check_condition(val, condition['threshold'], condition['direction']):
                secondary_match = False
                break
        
        if secondary_match: return mech_name

    return "UNCERTAIN_SIGNIFICANCE"

def extract_top_signals(row, delta_cols, k=3):
    """
    Extracts the top K signals by Magnitude, Gain, and Loss.
    Returns a Series to be assigned to new dataframe columns.
    """
    data = {c: row[c] for c in delta_cols}
    
    # 1. Top ABS (Magnitude)
    sorted_abs = sorted(data.items(), key=lambda x: abs(x[1]), reverse=True)[:k]
    abs_str = "; ".join([f"{key.replace('D_BED_', '')}={val:.3f}" for key, val in sorted_abs])
    top_abs = sorted_abs[0][1]
    
    # 2. Top GAIN (Positive) -> Descending
    gains = {key: val for key, val in data.items() if val > 0}
    sorted_gains = sorted(gains.items(), key=lambda x: x[1], reverse=True)[:k]
    gain_str = "; ".join([f"{key.replace('D_BED_', '')}={val:.3f}" for key, val in sorted_gains])
    
    # 3. Top LOSS (Negative) -> Ascending (most negative first)
    losses = {key: val for key, val in data.items() if val < 0}
    sorted_losses = sorted(losses.items(), key=lambda x: x[1])[:k] 
    loss_str = "; ".join([f"{key.replace('D_BED_', '')}={val:.3f}" for key, val in sorted_losses])
    
    return pd.Series([top_abs, abs_str, gain_str, loss_str])

# --- 3. EXECUTION ---

# Load Data
print("Loading data...")
final_df = pd.read_csv("variants_with_rationales.csv", index_col=0)

# Ensure numeric types for delta columns
# Assuming delta columns start at index 8 (adjust if needed or filter by name)
delta_cols = [c for c in final_df.columns if c.startswith("D_BED_")]
final_df[delta_cols] = final_df[delta_cols].apply(pd.to_numeric, errors='coerce')

# A. Assign Mechanisms
print("Assigning NT Mechanisms...")
final_df['NT_Mechanism'] = final_df.apply(
    lambda row: assign_mechanism(row, nt_hypothesis_logic), axis=1
)

# B. Extract Top Signals (New Feature)
print("Extracting Top Signals...")
final_df[['Top_abs_val','Top_Abs_Signals', 'Top_Gain_Signals', 'Top_Loss_Signals']] = final_df.apply(
    lambda row: extract_top_signals(row, delta_cols, k=3), axis=1
)

# C. Merge SongLab Consequence
print("Merging SongLab annotations...")
try:
    songlab = pd.read_parquet("hf://datasets/songlab/clinvar_vs_benign/test.parquet")
    # Ensure chromosome format matches (e.g., "chr1")
    songlab["chrom"] = "chr" + songlab["chrom"].astype(str).str.replace("chr", "") 
    
    # Merge
    merged = pd.merge(final_df, songlab, on=["chrom", "pos", "ref", "alt"], how="left")
    final_df["sognlab_consequence"] = merged["consequence"]
except Exception as e:
    print(f"Warning: Could not load SongLab dataset ({e}). Skipping that column.")
    final_df["sognlab_consequence"] = "Not Available"

# D. Reorder Columns
cols = final_df.columns.tolist()
meta_cols = cols[:8] # Adjust based on your file structure
new_cols = (
    meta_cols +
    ["NT_Mechanism", "sognlab_consequence"] +
    ["Top_Abs_Signals", "Top_Gain_Signals", "Top_Loss_Signals"] +
    [c for c in cols[8:] if c not in ["NT_Mechanism", "sognlab_consequence", "Top_Abs_Signals", "Top_Gain_Signals", "Top_Loss_Signals"]]
)

final_df = final_df[new_cols]

# 4. VIEW & SAVE
print("\nMechanism Counts:")
print(final_df['NT_Mechanism'].value_counts())

print("\nSample Output:")
print(final_df[['#VariationID', 'NT_Mechanism', 'Top_Abs_Signals']].head())

# final_df.to_csv("improved_variants_analyzed.csv")

Found 21 D_BED columns
Computing top_k_absolute...
Computing top_k_gain...
Computing top_k_loss...


,chrom,pos,ref,alt,label,top_k_absolute,top_k_gain,top_k_loss
0,chr1,69511,A,G,Benign,"{'stop_codon': -0.010524427518248558, 'always_...","{'always_on_exon': 0.007550835609436035, 'exon...","{'stop_codon': -0.010524427518248558, 'polyA_s..."
1,chr1,953279,T,C,Benign,"{'stop_codon': -0.013488655909895897, 'splice_...","{'splice_acceptor': 0.007707115262746811, 'int...","{'stop_codon': -0.013488655909895897, 'polyA_s..."
2,chr1,973858,G,C,Benign,"{'splice_donor': 0.05632144957780838, 'lncRNA'...","{'splice_donor': 0.05632144957780838, 'skipped...","{'lncRNA': -0.01701195538043976, 'start_codon'..."
3,chr1,973929,T,C,Benign,"{'intron': 0.026182539761066437, 'splice_donor...","{'intron': 0.026182539761066437, 'splice_donor...","{'always_on_exon': -0.020452499389648438, 'exo..."
4,chr1,978953,C,G,Benign,"{'stop_codon': -0.005807454697787762, 'enhance...","{'intron': 0.004100104793906212, '3UTR-': 0.00...","{'stop_codon': -0.005807454697787762, 'enhance..."


In [ ]:
import pandas as pd

nt_hypothesis_logic = {
    # --- GROUP A: SPLICING MECHANISMS ---
    "CANONICAL_SPLICE_LOSS": {
        "description": "Disruption of standard splice site leading to potential exon skipping or retention.",
        "primary": [
            {"feature": "D_BED_splice_donor", "threshold": -0.5, "direction": "negative"},
            {"feature": "D_BED_splice_acceptor", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [], # Strong enough to stand alone
        "priority": 1
    },
    
    "CRYPTIC_SPLICING_ACTIVATION": {
        "description": "Creation of a new splice site or activation of a cryptic site.",
        "primary": [
            {"feature": "D_BED_splice_donor", "threshold": 0.4, "direction": "positive"},
            {"feature": "D_BED_splice_acceptor", "threshold": 0.4, "direction": "positive"}
        ],
        "secondary": [
            {"feature": "D_BED_intron", "threshold": -0.2, "direction": "negative"} # Intron looks less like intron (pseudo-exon)
        ],
        "priority": 2
    },

    "EXON_SKIPPING": {
        "description": "Loss of exon recognition features.",
        "primary": [
             {"feature": "D_BED_exon", "threshold": -0.3, "direction": "negative"},
             {"feature": "D_BED_intron", "threshold": 0.3, "direction": "positive"} # Looks more like intron
        ],
        "secondary": [
             {"feature": "D_BED_splice_donor", "threshold": -0.2, "direction": "negative"} # Weak donor loss support
        ],
        "priority": 3
    },

    # --- GROUP B: TRANSLATION & CODING ---
    "START_CODON_LOSS": {
        "description": "Disruption of the translation initiation site.",
        "primary": [
            {"feature": "D_BED_start_codon", "threshold": -0.5, "direction": "negative"}
        ],
        "secondary": [
            {"feature": "D_BED_5UTR+", "threshold": 0.1, "direction": "any"} # Context shift in UTR
        ],
        "priority": 1
    },

    "PREMATURE_TRUNCATION_RISK": {
        "description": "Loss of ORF signal suggesting frameshift or nonsense mutation.",
        "primary": [
            {"feature": "D_BED_ORF", "threshold": -0.4, "direction": "negative"}
        ],
        "secondary": [
             {"feature": "D_BED_protein_coding_gene", "threshold": -0.2, "direction": "negative"}
        ],
        "priority": 4
    },

    # --- GROUP C: REGULATORY ---
    "INSULATOR_DYSFUNCTION": {
        "description": "Disruption of CTCF binding, potentially altering TAD boundaries.",
        "primary": [
            {"feature": "D_BED_CTCF-bound", "threshold": -0.4, "direction": "negative"}
        ],
        "secondary": [],
        "priority": 5
    },

    "TISSUE_SPECIFIC_SILENCING": {
        "description": "Loss of tissue-specific regulatory element without loss of housekeeping function.",
        "primary": [
            {"feature": "D_BED_promoter_Tissue_specific", "threshold": -0.3, "direction": "negative"},
            {"feature": "D_BED_enhancer_Tissue_specific", "threshold": -0.3, "direction": "negative"}
        ],
        "secondary": [
            # CRITICAL: Invariant promoter must NOT be lost (to distinguish from gene deletion)
            {"feature": "D_BED_promoter_Tissue_invariant", "threshold": -0.1, "direction": "stable"} 
        ],
        "priority": 6
    }
}

def check_condition(value, threshold, direction):
    """
    Helper to evaluate different direction types.
    """
    if direction == "negative":
        return value <= threshold
    elif direction == "positive":
        return value >= threshold
    elif direction == "stable":
        # Value must be within the noise range (+/- absolute threshold)
        return abs(value) < abs(threshold)
    elif direction == "any":
        # Magnitude is high, regardless of sign
        return abs(value) >= abs(threshold)
    return False


def assign_mechanism(row, hypothesis_logic):
    """
    Evaluates a single row of NTv3 deltas against a hypothesis logic dictionary.
    Returns the name of the first matching hypothesis (sorted by priority).
    """
    # 1. Sort hypotheses by priority (1 is highest, so we sort ascending)
    sorted_hypotheses = sorted(hypothesis_logic.items(), key=lambda x: x[1]['priority'])
    
    for mech_name, rules in sorted_hypotheses:
        # --- CHECK PRIMARY (OR Logic) ---
        # A hypothesis triggers if ANY primary feature meets its threshold
        primary_match = False
        for condition in rules['primary']:
            val = row.get(condition['feature'], 0) # Safety: default to 0 if col missing
            thresh = condition['threshold']
            direction = condition['direction']
            
            if check_condition(val, thresh, direction):
                primary_match = True
                break # One primary trigger is enough
        
        if not primary_match:
            continue # Skip to next hypothesis if no primary trigger found

        # --- CHECK SECONDARY (AND Logic) ---
        # If secondary conditions exist, ALL must be met to confirm the complex mechanism
        secondary_match = True
        for condition in rules['secondary']:
            val = row.get(condition['feature'], 0)
            thresh = condition['threshold']
            direction = condition['direction']
            
            if not check_condition(val, thresh, direction):
                secondary_match = False
                break
        
        # If both checks pass, return this mechanism immediately (respecting priority)
        if secondary_match:
            return mech_name

    return "UNCERTAIN_SIGNIFICANCE"



# --- USAGE ---

# 1. Load your dataframe (assuming it's already loaded as `df`)
# Ensure your delta columns are floats
final_df = pd.read_csv("variants_with_rationales.csv", index_col=0)


numeric_cols = final_df.columns[8:] # If columns 0-5 are metadata
final_df[numeric_cols] = final_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# 2. Apply the function
# We use a lambda to pass your specific dictionary structure
print("Assigning mechanisms...")
final_df['NT_Mechanism'] = final_df.apply(
    lambda row: assign_mechanism(row, nt_hypothesis_logic), 
    axis=1
)

cols = final_df.columns.tolist()




# 3. add songlab consequence
songlab = pd.read_parquet("hf://datasets/songlab/clinvar_vs_benign/test.parquet")
songlab["chrom"] = "chr" + songlab["chrom"].astype(str)
final_df["sognlab_consequence"] = pd.merge(final_df, songlab, on=["chrom", "pos", "ref", "alt"], how="left")["consequence"]

new_cols = (
    cols[:8] +
    ["NT_Mechanism"] + ["sognlab_consequence"] +
    [c for c in cols[8:] if c != "NT_Mechanism" and c != "sognlab_consequence"]
)

final_df = final_df[new_cols]

# 4. View the breakdown
print(final_df['NT_Mechanism'].value_counts())

# 5. Inspect a few examples
cols_to_show = ['#VariationID', 'NT_Mechanism'] + \
               ['D_BED_splice_donor', 'D_BED_intron', 'D_BED_start_codon'] # Context cols
print(final_df[final_df['NT_Mechanism'] != "UNCERTAIN_SIGNIFICANCE"][cols_to_show].head())


In [ ]:
import numpy as np
import pandas as pd

def create_aggregated_df(df, k=10):
    """
    Create aggregated dataframe with metadata columns and top-k aggregations.
    Assumes first 5 cols are metadata, rest include D_BED_* columns.
    """
    metadata_cols = df.columns[:5].tolist()

    # Filter for D_BED columns only (after metadata)
    d_bed_cols = [c for c in df.columns[5:] if "D_BED" in c]
    print(f"Found {len(d_bed_cols)} D_BED columns")

    def _get_numeric_values(row):
        # FORCE numeric (key fix)
        s = pd.to_numeric(row[d_bed_cols], errors="coerce")
        return s.dropna()

    def get_top_k_absolute(row):
        values = _get_numeric_values(row)
        if values.empty:
            return {}
        abs_values = values.abs()
        top = values.loc[abs_values.nlargest(min(k, len(values))).index]

        result = {col.replace("D_BED_", ""): float(val) for col, val in top.items()}
        abs_top = top.abs()
        result.update({
            "mean": float(abs_top.mean()),
            "median": float(abs_top.median()),
            "min": float(abs_top.min()),
            "max": float(abs_top.max()),
        })
        return result

    def get_top_k_gain(row):
        values = _get_numeric_values(row)
        if values.empty:
            return {}
        top = values.nlargest(min(k, len(values)))

        result = {col.replace("D_BED_", ""): float(val) for col, val in top.items()}
        result.update({
            "mean": float(top.mean()),
            "median": float(top.median()),
            "min": float(top.min()),
            "max": float(top.max()),
        })
        return result

    def get_top_k_loss(row):
        values = _get_numeric_values(row)
        if values.empty:
            return {}
        top = values.nsmallest(min(k, len(values)))

        result = {col.replace("D_BED_", ""): float(val) for col, val in top.items()}
        result.update({
            "mean": float(top.mean()),
            "median": float(top.median()),
            "min": float(top.min()),
            "max": float(top.max()),
        })
        return result

    agg_df = df[metadata_cols].copy()

    print("Computing top_k_absolute...")
    agg_df["top_k_absolute"] = df.apply(get_top_k_absolute, axis=1)

    print("Computing top_k_gain...")
    agg_df["top_k_gain"] = df.apply(get_top_k_gain, axis=1)

    print("Computing top_k_loss...")
    agg_df["top_k_loss"] = df.apply(get_top_k_loss, axis=1)

    return agg_df

# usage
k = 10
agg_df = create_aggregated_df(sample_df, k=k)
agg_df.head()


In [ ]:
import pandas as pd

# Load encode_experiments.tsv to get the column names
encode_exp = pd.read_csv("encode_experiments.tsv", sep="\t", nrows=1)
print("First row of encode_experiments.tsv:")
print(encode_exp.iloc[0].tolist()[:20], "...")  # Show first 20 values
print(f"\nTotal columns in encode_exp: {len(encode_exp.columns)}")